# GigaHands · 두 손 3D 포즈 학습 — HandLiteV3 (직선 맞춤 삼각측량 기준점 + 보정 신경망) · TPU v6e-1

카메라 3대의 비동기 이벤트(도착한 카메라 프레임의 2D 키포인트와 명목 ray)만 입력으로, 임의의 query 시각마다 두 손 관절 `[2,21,3]`과 관절별 예상 오차, 손별 '시야 안' 확률을 출력합니다. 설정 셀의 `ARCH`로 모델을 고릅니다: 기본 `'litev3'`(HandLiteV3) 또는 `'direct'`(HandDirect: 기하 계산 없이 신경망만, 비교용; 예상 오차·시야 판정·기준점 출력 없음).

**관절 상태 (배포·스트림):** 배포 런타임과 이벤트 단위 스트림은 관절마다 마지막 삼각측량(점과, 그때 쓴 최신 검출의 촬영 시각)을 기억합니다. 삼각측량이 성공할 때만 갱신하고 모델 예측은 넣지 않습니다. 상태는 과거 창 탐색(1.0초, 0.2초 간격)이 직전 위치를 **못 찾은 관절에만** 씁니다. 그래서 매 프레임 query해도 입력이 학습(`forward()`, 상태 없음)과 같고, 한 카메라에만 보이는 기간이 아무리 길어도 그 ray 위에서 마지막 삼각측량 깊이로 기준점을 잡습니다(종류 `ray`). 직전 위치 경과 시간 입력은 학습 범위 끝(1.2초)으로 잘라 넣습니다.

**시야 밖 손 (`PRESENCE`):** 손은 항상 있지만 세 카메라 시야를 모두 벗어날 수 있습니다.
- **라벨:** 클립마다 검출된 관절의 u·v와 명목 ray로 카메라별 픽셀 변환을 맞추고, **정답 3D 관절을 실제 카메라 자세로 투영**합니다. 관절의 절반 이상이 어느 한 카메라 화면 안(카메라 앞, [0,1] 범위)에 들어오는 손은 '시야 안', 아니면 '시야 밖'입니다(`data.hands_in_view`). 5번 셀이 맞춤 잔차와 시야 밖 비율을 출력합니다.
- **시야 밖 손:** 위치 손실이 없습니다. 대신 손마다 **'시야 안일 확률'**(`presence`, 토큰 gradient를 끊은 head)을 맞추도록 학습합니다(`PRESENCE_WEIGHT` 0.1).
- **시야 안인데 검출이 빠진 손:** 손이 분명히 거기 있으므로 위치도 계속 학습합니다.
- **지표:** MPJPE 등 위치 지표는 시야 안 손 기준입니다. 로그의 `in_view_acc`는 시야 판정 정확도, `out`은 시야 밖 손 비율입니다.
- **배포:** `runtime.in_view_probability`(손별)가 0.5보다 작으면 그 손은 세 카메라 모두 밖이니 좌표를 무시하면 됩니다.

## 모델 구조
1. **이벤트 도착 시 (프레임마다 한 번):** 관절별 u·v, ray 원점·방향, 촬영→도착 지연, 검출 여부 `[2,21,10]`를 손가락별 encoder에 넣어 손가락 토큰 10개(손 2 × 손가락 5) `[10,64]`를 만들어 저장합니다.
2. **query 시각 t마다: 기하 계산** (학습 파라미터 없음, float32, gradient 없음)
   - **ray:** 카메라 3대 × 관절 42개마다, 그 관절이 검출된 최근 0.2초 안의 프레임들의 ray 방향에 시간에 대한 직선을 최소제곱으로 맞추고 t에서 읽습니다. 검출이 하나면 그대로 씁니다.
   - **삼각측량:** 관절마다 카메라별 ray로 최소제곱 삼각측량합니다.
   - **직전 위치:** 0.2~1.0초 전(0.2초 간격, `prior_step_s`, `prior_span_s`; 각 0.2초 맞춤 구간이 겹치지 않음)에 같은 직선 맞춤 + 삼각측량을 해서, 그 관절이 성공한 가장 최근 위치를 찾습니다. 배포·스트림에서는 여기서 못 찾은 관절만 기억한 마지막 삼각측량을 씁니다.
   - **지금 실패한 관절의 기준점:**
     - 그 관절을 본 카메라가 있으면: 보이는 ray 중 직전 위치에 가장 가까이 지나는 ray 위에서, 직전 위치와 가장 가까운 점. 방향은 지금 관측, 깊이는 과거에서 옵니다.
     - 본 카메라가 없으면: 직전 위치
     - 직전 위치도 없으면(학습: 최근 1.2초 동안 한 번도 삼각측량 안 됨, 배포: 한 번도 삼각측량된 적 없음): 0 (보정 신경망이 위치 전체를 냄)
3. **관절 토큰 입력 56개**
   - 관절 11개: 기준점 3, 기준점 − 직전 위치 3, 기준점 종류(삼각측량 / ray + 직전 깊이 / 직전 위치) 3, 직전 위치가 있는지 1, 직전 위치의 경과 시간 1
   - 카메라별 15개 × 3: 검출 여부, 기울기 사용 여부, 검출 수, 최신·가장 오래된 검출 경과 시간, 최신 u·v 2, ray와 기준점의 어긋남 3, 맞춘 ray와 최신 ray의 차이 3(**맞춤이 과했는지**), 직선에서 벗어난 정도 RMS·최신 검출(**검출이 튀는지**)
4. **보정 신경망 (corrector)**
   - 관절 토큰마다 MLP(56→64→64) + 관절별 학습 embedding
   - 블록 × 2: 카메라별 최근 8개 이벤트(0.5초 안)의 손가락 토큰 240개를 참조(경과 시간 embedding, 빈 null 토큰) → 양손 42개 관절끼리 self-attention → feed-forward
   - 출력: 관절별 보정값 `[42,3]`(0 초기화: 학습 전에는 기준점 그대로)과 예상 오차 `[42]`(log(1+mm), 토큰 gradient를 끊은 별도 head)
5. **최종 좌표 = 기준점 + 보정값**
6. **손실:** 포즈(SmoothL1 β 6mm + 뼈 길이 0.1), 손목 기준 상대 좌표 1.0, 예상 오차 0.1

## 로그
- epoch 줄: MPJPE, PCK20, `rel`(손 모양), `error_miss`(예상 오차가 실제에서 평균 몇 mm 빗나가는지)
- `kinds` 줄: 기준점 종류(`none`, `triangulated`, `ray`, `prior`)별 비율 / MPJPE / (보정 전 기준점만의 오차). 괄호 안과의 차이가 보정 신경망의 효과입니다.

## TPU 학습
- 데이터 로딩이 CPU 병목이라 TPU 호스트 코어를 다 씁니다(worker = 코어 수 − 2, worker마다 스레드 1개, `PREFETCH`개씩 미리 준비). 윈도 이벤트 선택은 이진 탐색으로 필요한 구간만 봅니다.
- float32, 고정 배치 shape(이벤트 `MAX_EVENTS`개로 패딩, 학습 epoch의 마지막 불완전 배치는 버림). 첫 배치는 XLA 컴파일로 느립니다.
- 체크포인트는 CPU 텐서로 저장하므로 CPU 예측·내보내기에서 그대로 열립니다.

## 배포
ncnn 그래프 2개(`encoder` 이벤트 도착 시, `corrector` query 시)와 numpy 런타임(`hand_tracking/runtime.py`: 직선 맞춤, 삼각측량, 직전 위치 탐색, 기준점, 관절 입력, 슬롯 선택)만 씁니다. `runtime.error_mm`으로 관절별 예상 오차(mm), `runtime.anchor_kind`로 기준점 종류를 읽습니다.

## 먼저 할 일
1. Colab에서 이 `.ipynb`를 엽니다.
2. **런타임 → 런타임 유형 변경 → TPU (v6e-1)**를 선택합니다. PyTorch/XLA가 없으면 1번 셀이 설치합니다.
3. PC의 `exports/gigahands_pi3_overlap.zip`을 Google Drive의 `MyDrive/GigaHands/`에 올립니다. ZIP 안에는 `manifest.jsonl`, `train/`, `val/`가 있어야 합니다.
4. 설정 셀의 경로와 실행 이름을 확인한 뒤 위에서부터 실행합니다.

데이터: 원본 클립 12,673개, 학습 9,628개/검증 1,078개, 참가자 44명/6명. ZIP 약 6.92GB, 압축 해제 후 약 7.00GB입니다.


In [ ]:
# 1. 환경 확인 — TPU 런타임(v6e-1)의 PyTorch/XLA. 없으면 설치합니다.
# TPU는 한 프로세스만 쓸 수 있어 이 노트북 커널은 TPU를 잡지 않고, 학습·확인은 하위 프로세스에서 실행합니다.
import sys, os, json, hashlib, shutil, subprocess, importlib.util, importlib.metadata
from pathlib import Path, PurePosixPath

os.environ.setdefault('PJRT_DEVICE', 'TPU')    # 하위 프로세스(학습)에 물려줌
# 데이터 worker가 많으므로 각 프로세스의 numpy(BLAS) 스레드는 1개: 코어를 worker끼리 나눠 쓰고 서로 빼앗지 않게
for name in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS'):
    os.environ.setdefault(name, '1')
LIBTPU_INDEXES = ['-f', 'https://storage.googleapis.com/libtpu-wheels/index.html',
                  '-f', 'https://storage.googleapis.com/libtpu-releases/index.html']
FALLBACK_VERSION = '2.8.0'   # 설치된 torch에 맞는 torch_xla가 없을 때 함께 설치할 짝 버전

def pip(*packages):
    return subprocess.run([sys.executable, '-m', 'pip', 'install', *packages, *LIBTPU_INDEXES]).returncode == 0

if importlib.util.find_spec('torch_xla') is None:
    # torch를 import하기 전에 설치해야 버전을 바꿔도 런타임을 다시 시작할 필요가 없습니다.
    try:
        torch_version = importlib.metadata.version('torch').split('+')[0]
    except importlib.metadata.PackageNotFoundError:
        torch_version = None
    if not (torch_version and pip(f'torch_xla[tpu]=={torch_version}')):
        assert pip(f'torch=={FALLBACK_VERSION}', f'torch_xla[tpu]=={FALLBACK_VERSION}'), 'PyTorch/XLA 설치 실패'
    importlib.invalidate_caches()

missing = [name for name in ('numpy', 'matplotlib', 'tqdm') if importlib.util.find_spec(name) is None]
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', *missing], check=True)
import torch
import numpy as np
from tqdm.auto import tqdm
assert tuple(map(int, torch.__version__.split('+')[0].split('.')[:2])) >= (2, 5), 'PyTorch 2.5 이상이 필요합니다.'
probe = subprocess.run([sys.executable, '-c', (
    'import torch, torch_xla, torch_xla.core.xla_model as xm\n'
    'd = torch_xla.device() if hasattr(torch_xla, "device") else xm.xla_device()\n'
    'print("torch_xla:", torch_xla.__version__)\n'
    'print("device:", d, xm.xla_device_hw(d))\n'
    'print("check:", float((torch.ones(4, 4, device=d) @ torch.ones(4, 4, device=d)).sum()))')],
    capture_output=True, text=True)
print(probe.stdout.strip())
assert probe.returncode == 0 and 'TPU' in probe.stdout, (
    'TPU를 쓸 수 없습니다. 런타임 유형을 TPU(v6e-1)로 바꾸고 다시 실행하세요.\n' + probe.stderr[-2000:])
print('PyTorch:', torch.__version__, '| CPU cores:', os.cpu_count())
WORK_DIR = Path('/content/gigahands_training')
WORK_DIR.mkdir(exist_ok=True)
for package in ('hand_tracking', 'training'):
    (WORK_DIR / package).mkdir(exist_ok=True)
os.chdir(WORK_DIR)
if str(WORK_DIR) not in sys.path:
    sys.path.insert(0, str(WORK_DIR))


In [ ]:
# 2. Drive 연결
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# 3. 사용자 설정
DRIVE_ZIP = Path('/content/drive/MyDrive/GigaHands/gigahands_pi3_overlap.zip')
DRIVE_RUNS = Path('/content/drive/MyDrive/GigaHands/runs')
RUN_NAME = 'event_litev3_tpu_e85_v4'   # 소스가 바뀐 노트북은 새 이름으로 학습 (5번 셀의 해시 검사)
ARCH = 'litev3'                   # 'litev3': HandLiteV3 (기본) / 'direct': HandDirect (신경망만, 비교용)
DEVICE = 'xla'                    # TPU (PyTorch/XLA). 예측·내보내기는 CPU
EPOCHS = 85                       # 배치 128: 배치 64·60 epoch보다 스텝이 적어 epoch로 보충
BATCH_SIZE = 128                  # × TRAIN_QUERIES = 스텝당 query 512개
LEARNING_RATE = 8.5e-4            # 배치의 제곱근에 비례: 64일 때 6e-4, 128일 때 8.5e-4
WINDOWS_PER_CLIP = 16             # epoch마다 클립당 무작위 구간; 0이면 전체
TRAIN_QUERIES = 4                 # 학습은 윈도의 마지막 query 4개만(평가 지표는 원래 마지막 query); 0이면 16개 전부
VAL_WINDOWS_PER_CLIP = 16         # 매 epoch 고정 검증 구간; 0이면 전체
# 데이터 로딩이 CPU 병목이라 TPU 호스트의 코어를 다 씁니다: 메인 프로세스와 TPU 전송 스레드 몫 2개를
# 빼고 모두 worker. worker마다 스레드 1개, 배치 PREFETCH개씩 미리 준비
WORKERS = max(2, (os.cpu_count() or 4)-2)
PREFETCH = 4
THREADS = 4                       # 학습 메인 프로세스의 PyTorch CPU 스레드
SEED = 42
MODEL_OPTIONS = {
    'litev3': {
        '--dim': 64, '--heads': 4,
        '--blocks': 2,                         # 보정 신경망 블록 (손가락 토큰 참조 → 관절끼리 → feed-forward)
        '--fit-span-s': 0.2,                   # 기준점: 카메라·관절마다 최근 0.2초의 검출에 직선을 맞춰 query 시각에서 읽고 삼각측량
        '--prior-span-s': 1.0, '--prior-step-s': 0.2,        # 실패한 관절: 0.2~1.0초 전(0.2초 간격)에 같은 방식으로 삼각측량한 가장 최근 성공 위치를 씀
                                                             # (배포·스트림은 이보다 오래된 마지막 삼각측량도 상태로 기억해 씀)
        '--slots-per-camera': 8, '--event-span-s': 0.5,          # 손가락 토큰: 카메라당 최근 이벤트 8개, 0.5초 안
    },
    'direct': {'--dim': 64, '--heads': 4, '--fusion-blocks': 2, '--blocks': 2, '--slots-per-camera': 8,
               '--event-span-s': 0.5},
}[ARCH]
# 한 구간이 담는 최대 이벤트 수. 과거 1.45초(직전 위치 탐색 1.0초 + 직선 맞춤 0.2초 + 도착 여유 0.25초)라
# 약 80~90개로 예상해 128. 고정 shape라 패딩까지 계산하므로 너무 크게 두지 않습니다. 99%가 넘치면 5번 셀이 경고합니다.
MAX_EVENTS = 128
DROPOUT = 0.1
# 손은 항상 있지만 세 카메라 시야를 모두 벗어날 수 있습니다. 정답 3D 관절을 실제 카메라로 투영해 절반 이상이 어느
# 카메라 화면 안에 들어오는 손만 '시야 안'으로 보고, 시야 밖 손은 위치 손실 없이 '시야 밖'이라는 것만 학습합니다.
PRESENCE = True                   # 손마다 '시야 안일 확률' 출력 (토큰의 gradient를 끊어 포즈 학습에는 영향 없음)
PRESENCE_WEIGHT = 0.1             # 시야 안/밖 손실 가중치
ERROR_ESTIMATE = True             # 관절마다 '내 예측이 몇 mm 틀렸을지' 출력. 토큰의 gradient를 끊어 포즈 학습에는 영향 없음
# 손실 가중치 (정답은 rig 좌표계뿐: 참 월드 좌표 손실 없음, 카메라 보정 신경망 없음)
RELATIVE_WEIGHT = 1.0             # 손목 기준 상대 좌표 손실 (손 모양; 손목 위치 오차와 분리)
ERROR_WEIGHT = 0.1                # 예상 오차 손실
MODEL_ARGS = (['--arch', ARCH, '--dropout', DROPOUT, '--max-events', MAX_EVENTS]
              + [value for item in MODEL_OPTIONS.items() for value in item]
              + ([] if ERROR_ESTIMATE or ARCH != 'litev3' else ['--no-error-estimate'])
              + ([] if PRESENCE or ARCH != 'litev3' else ['--no-presence']))   # HandDirect에는 두 출력이 없음
LOSS_ARGS = ['--relative-weight', RELATIVE_WEIGHT, '--error-weight', ERROR_WEIGHT, '--presence-weight', PRESENCE_WEIGHT]

assert RUN_NAME and Path(RUN_NAME).name == RUN_NAME and RUN_NAME not in ('.', '..')
assert ARCH in ('litev3', 'direct')
assert EPOCHS > 0 and BATCH_SIZE > 0 and WORKERS >= 0 and MAX_EVENTS > 0
assert LEARNING_RATE > 0 and WINDOWS_PER_CLIP >= 0 and VAL_WINDOWS_PER_CLIP >= 0 and TRAIN_QUERIES >= 0
assert RELATIVE_WEIGHT >= 0 and ERROR_WEIGHT >= 0 and PRESENCE_WEIGHT >= 0
assert DRIVE_ZIP.is_file(), f'Drive에 데이터 ZIP을 올리고 경로를 확인하세요: {DRIVE_ZIP}'
LOCAL_ZIP = Path('/content/gigahands_pi3_overlap.zip')
EXTRACT_DIR = Path('/content/gigahands_dataset')
CACHE_DIR = Path('/content/gigahands_cache')   # 첫 epoch에 전처리한 클립(압축 없음); 이후 epoch는 여기서 읽음
EXPERIMENT_DIR = DRIVE_RUNS / RUN_NAME
RUN_DIR = EXPERIMENT_DIR / 'checkpoints'
print('학습 결과:', RUN_DIR)


## 데이터 복사 및 검사
처음 실행할 때 데이터 ZIP을 복사·해제합니다. 연결이 끊겨 런타임 로컬 파일이 사라지면 이 셀을 다시 실행하세요. Drive의 원본 ZIP과 체크포인트는 보존됩니다. 진행 막대가 표시됩니다.


In [ ]:
# 4. ZIP 로컬 복사 + 안전한 압축 해제 (기존 부분 해제는 다시 완료)
import zipfile, stat
EXPECTED_MANIFEST_SHA256 = 'b4e950b05438ffb5ad7ee846918cc553dfa1d9ff832386ce5b527003916816fc'
zip_size = DRIVE_ZIP.stat().st_size
if not LOCAL_ZIP.is_file() or LOCAL_ZIP.stat().st_size != zip_size:
    assert shutil.disk_usage('/content').free > zip_size + 7001743710 + 1_000_000_000, '로컬 디스크 공간이 부족합니다.'
    partial = LOCAL_ZIP.with_suffix('.zip.part')
    with DRIVE_ZIP.open('rb') as src, partial.open('wb') as dst, tqdm(total=zip_size, unit='B', unit_scale=True, desc='Copy ZIP') as bar:
        while chunk := src.read(8 * 1024 * 1024):
            dst.write(chunk)
            bar.update(len(chunk))
    partial.replace(LOCAL_ZIP)

EXTRACT_DIR.mkdir(exist_ok=True)
ready = EXTRACT_DIR / '.ready.json'
reuse = ready.is_file() and json.loads(ready.read_text()).get('manifest_sha256') == EXPECTED_MANIFEST_SHA256
if not reuse:
    with zipfile.ZipFile(LOCAL_ZIP) as archive:
        infos = archive.infolist()
        assert shutil.disk_usage('/content').free > sum(i.file_size for i in infos) + 1_000_000_000, '압축 해제 공간이 부족합니다.'
        seen = set()
        for info in infos:
            relative = PurePosixPath(info.filename.replace('\\', '/'))
            target = (EXTRACT_DIR / str(relative)).resolve()
            if relative.is_absolute() or '..' in relative.parts or not target.is_relative_to(EXTRACT_DIR.resolve()):
                raise ValueError(f'잘못된 ZIP 경로: {info.filename}')
            if stat.S_ISLNK(info.external_attr >> 16) or str(relative) in seen:
                raise ValueError(f'링크 또는 중복 ZIP 경로: {info.filename}')
            seen.add(str(relative))
        for info in tqdm(infos, desc='Extract ZIP'):
            relative = PurePosixPath(info.filename.replace('\\', '/'))
            target = EXTRACT_DIR / str(relative)
            if info.is_dir():
                target.mkdir(parents=True, exist_ok=True)
                continue
            target.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(info) as src, target.open('wb') as dst:
                shutil.copyfileobj(src, dst, length=4 * 1024 * 1024)

manifests = list(EXTRACT_DIR.rglob('manifest.jsonl'))
assert len(manifests) == 1, 'manifest.jsonl이 하나인 데이터 ZIP이 필요합니다.'
DATA_DIR = manifests[0].parent
manifest_hash = hashlib.sha256(manifests[0].read_bytes()).hexdigest()
assert manifest_hash == EXPECTED_MANIFEST_SHA256, '이 노트북에 연결된 균등 데이터셋과 ZIP이 다릅니다.'
ready.write_text(json.dumps({'manifest_sha256': manifest_hash}))
print('DATA_DIR:', DATA_DIR)


## 모델 소스
아래 25개 셀은 모델·로더·학습·평가·예측·ncnn 내보내기 코드를 로컬에 씁니다. 보통 수정할 필요가 없습니다. 5번 셀이 각 파일의 해시를 확인하므로, 코드를 바꾸는 실험은 노트북과 실행 이름을 새로 만드세요.
- `training/`: `train.py`, `evaluate.py`, `predict.py`, `export.py` (명령줄 진입점)
- `hand_tracking/model.py`: HandLiteV3 (직선 맞춤 ray, 삼각측량, 직전 위치로 실패 관절의 기준점, 관절 입력, 보정 신경망), 이벤트 단위 스트림
- `hand_tracking/geometry.py`: 최소제곱 삼각측량, ray와 점의 각도 잔차
- `hand_tracking/networks.py`, `layers.py`: encoder 입력, attention 블록, 손가락 토큰 encoder
- `hand_tracking/events.py`, `data.py`, `stream.py`: 이벤트 수락·슬롯 선택, NPZ 구간 로더, 스트리밍 버퍼
- `hand_tracking/engine.py`, `objectives.py`: epoch 실행, 손실·지표
- `hand_tracking/deploy.py`, `runtime.py`: ncnn 내보내기, PyTorch 없는 numpy 런타임
- `hand_tracking/direct.py`, `direct_export.py`, `direct_runtime.py`: HandDirect (비교용) 모델·내보내기·런타임
- `hand_tracking/accelerator.py`: PyTorch/XLA 장치·동기화·CPU 저장


In [ ]:
%%writefile /content/gigahands_training/training/__init__.py
"""Command-line entry points: python -m training.<train|evaluate|predict|export>."""


In [ ]:
%%writefile /content/gigahands_training/training/train.py
"""Train a pose model on GigaHands camera-event windows: HandLiteV3 (default) or the
networks-only comparison HandDirect (--arch direct)."""
import argparse
import hashlib
import json
import math
from pathlib import Path
import torch
from torch.utils.data import DataLoader
from hand_tracking.accelerator import is_xla, manual_seed, xla_device
from hand_tracking.checkpoints import build_model, restore_training, save_checkpoint, training_checkpoint
from hand_tracking.config import (ARCHITECTURES, DEFAULT_ARCHITECTURE, SamplingConfig, add_model_arguments,
                                  model_config_from_args)
from hand_tracking.data import HandWindows
from hand_tracking.engine import run_epoch

# Settings that must be > 0 and >= 0; the model's own fields are checked by its config.
POSITIVE = ('epochs', 'batch_size', 'threads', 'lr', 'prefetch')
NONNEGATIVE = ('workers', 'seed', 'max_clips', 'max_train_batches', 'max_val_batches', 'bone_weight', 'train_queries',
               'weight_decay', 'relative_weight', 'error_weight', 'presence_weight')
# Settings that change speed, not results: a resume may differ in them.
RUNTIME = ('workers', 'threads', 'cache_dir', 'prefetch')


def build_parser(argv=None):
    """The model flags depend on --arch, so it is read first."""
    first = argparse.ArgumentParser(add_help=False)
    first.add_argument('--arch', choices=sorted(ARCHITECTURES), default=DEFAULT_ARCHITECTURE)
    architecture = first.parse_known_args(argv)[0].arch
    p = argparse.ArgumentParser(description=__doc__)
    p.add_argument('--arch', choices=sorted(ARCHITECTURES), default=DEFAULT_ARCHITECTURE,
                   help='litev3: HandLiteV3; direct: HandDirect (networks only, comparison)')
    p.add_argument('--data', required=True)
    p.add_argument('--output', required=True)
    p.add_argument('--epochs', type=int, default=85)
    p.add_argument('--batch-size', type=int, default=128)
    add_model_arguments(p, ARCHITECTURES[architecture])
    p.add_argument('--max-events', type=int, default=SamplingConfig.max_events,
                   help='Event slots per window (oldest dropped beyond)')
    p.add_argument('--target-frame', choices=('rig', 'world'), default='rig',
                   help="Pose targets in the placed rig's frame (rig-wide error removed) or the true world")
    p.add_argument('--keep-out-of-view', action='store_true',
                   help='Keep position targets of hands outside all cameras (default: only the in-view loss teaches them)')
    p.add_argument('--relative-weight', type=float, default=1.,
                   help='Weight of the wrist-relative pose loss (hand shape, free of the wrist position error)')
    p.add_argument('--error-weight', type=float, default=.1,
                   help='Weight of the joint error estimate loss (never changes the pose)')
    p.add_argument('--presence-weight', type=float, default=.1,
                   help='Weight of the hand in-view loss (never changes the pose)')
    p.add_argument('--lr', type=float, default=8.5e-4)
    p.add_argument('--weight-decay', type=float, default=.01)
    p.add_argument('--bone-weight', type=float, default=.1)
    p.add_argument('--windows-per-clip', type=int, default=16)
    p.add_argument('--train-queries', type=int, default=4,
                   help="Train on each window's last N queries only (0: all); validation keeps all")
    p.add_argument('--val-windows-per-clip', type=int, default=16)
    p.add_argument('--workers', type=int, default=0)
    p.add_argument('--prefetch', type=int, default=4, help='Batches each data worker prepares ahead')
    p.add_argument('--cache-dir', default='',
                   help='Folder for uncompressed preprocessed clips, written on first read (about 2x the data size)')
    p.add_argument('--threads', type=int, default=4)
    p.add_argument('--seed', type=int, default=42)
    p.add_argument('--device', choices=['auto', 'cpu', 'cuda', 'xla'], default='auto',
                   help='xla: TPU through PyTorch/XLA (float32, fixed batch shapes)')
    p.add_argument('--max-clips', type=int, default=0, help='Development subset per split; 0 means all')
    p.add_argument('--max-train-batches', type=int, default=0)
    p.add_argument('--max-val-batches', type=int, default=0)
    p.add_argument('--resume', action='store_true', help='Resume output/last.pt with identical settings')
    return p


def parse_settings(parser, argv=None):
    """Arguments plus validated model and sampling configs; errors exit through the parser."""
    args = parser.parse_args(argv)
    if min(getattr(args, name) for name in POSITIVE) <= 0 or not math.isfinite(args.lr):
        parser.error('Invalid training settings')
    if min(getattr(args, name) for name in NONNEGATIVE) < 0:
        parser.error('Settings must be nonnegative')
    try:
        return args, model_config_from_args(args, ARCHITECTURES[args.arch]), SamplingConfig(max_events=args.max_events, target_frame=args.target_frame,
                                                                    mask_out_of_view=not args.keep_out_of_view)
    except ValueError as error:
        parser.error(str(error))


def select_device(name):
    if name == 'auto':
        return torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    if name == 'cuda' and not torch.cuda.is_available():
        raise RuntimeError('CUDA is unavailable in this PyTorch installation')
    if name == 'xla':
        return xla_device()
    return torch.device(name)


def record_run(parser, args, output):
    """Write run.json, or on resume check it matches; returns the recorded options."""
    options = vars(args).copy()
    options.pop('resume')
    options['manifest_sha256'] = hashlib.sha256((Path(args.data)/'manifest.jsonl').read_bytes()).hexdigest()
    path = output/'run.json'
    if args.resume:
        previous = json.loads(path.read_text())
        # Flags added after that run was started match when they are at their defaults.
        previous.update({k: v for k, v in options.items() if k not in previous and v == parser.get_default(k)})
        if ({k: v for k, v in previous.items() if k not in RUNTIME}
                != {k: v for k, v in options.items() if k not in RUNTIME}):
            raise ValueError('Resume settings/dataset differ')
    else:
        path.write_text(json.dumps(options, indent=2), encoding='utf-8')
    return options


def build_loaders(args, config, sampling, device):
    train = HandWindows(args.data, 'train', windows_per_clip=args.windows_per_clip, seed=args.seed,
                        max_clips=args.max_clips,
                        context_s=config.context_s, max_events=sampling.max_events, target_frame=sampling.target_frame,
                        queries=args.train_queries, cache=args.cache_dir or None, mask_out_of_view=sampling.mask_out_of_view)
    val = HandWindows(args.data, 'val', windows_per_clip=args.val_windows_per_clip, seed=args.seed,
                      max_clips=args.max_clips, context_s=config.context_s, max_events=sampling.max_events,
                      target_frame=sampling.target_frame, cache=args.cache_dir or None,
                      mask_out_of_view=sampling.mask_out_of_view)
    # XLA compiles once per batch shape: training drops the last, smaller batch of an epoch
    # (validation keeps it, one extra shape compiled once and reused). With many workers each
    # keeps one thread (no oversubscription of the cores) and prefetches several batches.
    # Workers are not persistent: each epoch's workers see the dataset's new epoch (shuffle).
    extra = dict(worker_init_fn=_single_thread, prefetch_factor=args.prefetch) if args.workers else {}
    return [DataLoader(dataset, batch_size=args.batch_size, num_workers=args.workers,
                       pin_memory=device.type == 'cuda', drop_last=is_xla(device) and dataset is train, **extra)
            for dataset in (train, val)]


def _single_thread(worker_id):
    """DataLoader worker: one PyTorch thread (numpy's BLAS threads are set by the parent's
    OMP_NUM_THREADS/OPENBLAS_NUM_THREADS)."""
    torch.set_num_threads(1)


def main(argv=None):
    parser = build_parser(argv)
    args, config, sampling = parse_settings(parser, argv)
    torch.set_num_threads(args.threads)
    torch.manual_seed(args.seed)
    device = select_device(args.device)
    manual_seed(device, args.seed)
    output = Path(args.output)
    output.mkdir(parents=True, exist_ok=args.resume)
    options = record_run(parser, args, output)
    train_loader, val_loader = build_loaders(args, config, sampling, device)
    model = build_model(config).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=args.epochs)
    scaler = torch.amp.GradScaler('cuda', enabled=device.type == 'cuda')
    first, best = 0, float('inf')
    if args.resume:
        first, best = restore_training(output/'last.pt', model, optimizer, scheduler, scaler)
    print(f'device={device} parameters={sum(p.numel() for p in model.parameters()):,} '
          f'train_windows/epoch<={len(train_loader.dataset):,} val_windows<={len(val_loader.dataset):,} '
          f'(events from {config.context_s:.2f} s before the first query)', flush=True)
    if args.max_clips or args.max_train_batches or args.max_val_batches:
        print('DEVELOPMENT RUN: metrics are not full-dataset performance.', flush=True)
    losses = dict(bone_weight=args.bone_weight, relative_weight=args.relative_weight, error_weight=args.error_weight,
                  presence_weight=args.presence_weight)
    for epoch in range(first, args.epochs):
        train_loader.dataset.epoch = epoch
        train = run_epoch(model, train_loader, device, optimizer, scaler, args.max_train_batches, **losses)
        val = run_epoch(model, val_loader, device, limit=args.max_val_batches, **losses)
        scheduler.step()
        improved = val['mpjpe_mm'] < best
        best = min(best, val['mpjpe_mm'])
        with (output/'metrics.jsonl').open('a', encoding='utf-8') as f:
            f.write(json.dumps(dict(epoch=epoch+1, train=train, val=val))+'\n')
        saved = training_checkpoint(model, sampling, optimizer, scheduler, scaler, epoch, best, options)
        save_checkpoint(output/'last.pt', saved)
        if improved:
            save_checkpoint(output/'best.pt', saved)
        error_miss = '' if val['error_miss_mm'] is None else f" error_miss={val['error_miss_mm']:.2f}mm"
        if val['presence_accuracy'] is not None:
            error_miss += f" in_view_acc={val['presence_accuracy']*100:.1f}% (out {val['out_of_view_rate']*100:.1f}%)"
        print(f"epoch={epoch+1}/{args.epochs} train={train['mpjpe_mm']:.2f}mm val={val['mpjpe_mm']:.2f}mm "
              f"PCK20={val['pck20']:.3f} rel={val['mpjpe_rel_mm']:.2f}mm{error_miss}", flush=True)
        # Per anchor kind: share of the joints with a target, their MPJPE and (anchored kinds) the
        # anchor's alone on the same joints (final query)
        def kind_text(name):
            text = f"{name}={val['kind_rate'][name]*100:.1f}%"
            if val['kind_mpjpe_mm'][name] is not None:
                text += f"/{val['kind_mpjpe_mm'][name]:.1f}mm"
            alone = val['kind_anchor_mpjpe_mm'].get(name)
            return text if alone is None else text+f"(anchor {alone:.1f}mm)"
        if val['kind_rate']['triangulated'] is not None:        # models with an anchor
            print('  kinds '+' '.join(kind_text(name) for name in val['kind_rate']), flush=True)


if __name__ == '__main__':
    main()


In [ ]:
%%writefile /content/gigahands_training/training/evaluate.py
"""Evaluate a checkpoint (HandLiteV3 or HandDirect) on fixed validation windows."""
import argparse
from dataclasses import replace
import json
from pathlib import Path
import torch
from torch.utils.data import DataLoader
from hand_tracking.accelerator import xla_device
from hand_tracking.checkpoints import load_checkpoint
from hand_tracking.data import HandWindows
from hand_tracking.engine import run_epoch


def parse_args(argv=None):
    p = argparse.ArgumentParser(description=__doc__)
    p.add_argument('--split', choices=['val', 'test'], default='val', help='Use test only after model selection')
    p.add_argument('--checkpoint', required=True)
    p.add_argument('--data', required=True)
    p.add_argument('--output', required=True)
    p.add_argument('--batch-size', type=int, default=16)
    p.add_argument('--windows-per-clip', type=int, default=16, help='0 evaluates all validation windows')
    p.add_argument('--workers', type=int, default=0)
    p.add_argument('--device', choices=['cpu', 'cuda', 'xla'], default='cpu')
    p.add_argument('--max-clips', type=int, default=0, help='Development subset; 0 uses all clips')
    p.add_argument('--max-events', type=int, help='Override the checkpoint input layout')
    return p.parse_args(argv)


def main(argv=None):
    args = parse_args(argv)
    torch.set_num_threads(4)
    device = xla_device() if args.device == 'xla' else torch.device(args.device)
    checkpoint = load_checkpoint(args.checkpoint, device)
    config, sampling = checkpoint.model.config, checkpoint.sampling
    if args.max_events is not None:
        sampling = replace(sampling, max_events=args.max_events)
    dataset = HandWindows(args.data, args.split, args.windows_per_clip, max_clips=args.max_clips,
                          context_s=config.context_s, max_events=sampling.max_events, target_frame=sampling.target_frame,
                          mask_out_of_view=sampling.mask_out_of_view)
    loader = DataLoader(dataset, batch_size=args.batch_size, num_workers=args.workers)
    options = checkpoint.saved['training_options']
    report = run_epoch(checkpoint.model, loader, device, bone_weight=options['bone_weight'],
                       relative_weight=options.get('relative_weight', 0.), error_weight=options.get('error_weight', 0.),
                       presence_weight=options.get('presence_weight', 0.))
    report.update(checkpoint=str(Path(args.checkpoint).resolve()), split=args.split, windows_per_clip=args.windows_per_clip,
                  max_events=sampling.max_events, target_frame=sampling.target_frame,
                  evaluated_clips=len(dataset.rows), limited_clips=bool(args.max_clips))
    with Path(args.output).open('x', encoding='utf-8') as f:
        json.dump(report, f, indent=2)
    print(json.dumps(report), flush=True)


if __name__ == '__main__':
    main()


In [ ]:
%%writefile /content/gigahands_training/training/predict.py
"""Predict one NPZ window from a trained checkpoint."""
import argparse
from pathlib import Path
import numpy as np
import torch
from hand_tracking.checkpoints import load_checkpoint
from hand_tracking.contracts import model_inputs
from hand_tracking.data import read_clip, make_sample


def parse_args(argv=None):
    p = argparse.ArgumentParser(description=__doc__)
    p.add_argument('--checkpoint', required=True)
    p.add_argument('--input', required=True)
    p.add_argument('--output', required=True)
    p.add_argument('--window-index', type=int, default=-1)
    return p.parse_args(argv)


def main(argv=None):
    args = parse_args(argv)
    torch.set_num_threads(4)
    checkpoint = load_checkpoint(args.checkpoint)
    model = checkpoint.model
    clip = read_clip(args.input)
    window = args.window_index % len(clip['window_starts'])
    sample = make_sample(clip, window, context_s=model.config.context_s, max_events=checkpoint.sampling.max_events,
                         target_frame=checkpoint.sampling.target_frame, mask_out_of_view=checkpoint.sampling.mask_out_of_view)
    if not sample['event_present'].any():
        raise ValueError('No arrived observations in this window')
    with torch.inference_mode():
        output = model(*(value[None] for value in model_inputs(sample)), return_details=True)
    if not bool(output.accepted[0, -1]):
        raise ValueError('The model used no event for the final query of this window')
    pose = output.pose[0, -1].numpy()
    # Each joint's anchor and how it was anchored (model.ANCHOR_KINDS; HandLiteV3 only), and its expected error in mm.
    extra = {}
    if output.anchored is not None:
        extra.update(anchor_xyz=output.anchor_xyz[0, -1].numpy(), anchor_kind=output.anchored[0, -1].numpy())
    if output.error is not None:
        extra['expected_error_mm'] = torch.expm1(output.error[0, -1]).numpy()
    if output.presence is not None:                           # each hand: probability of being in some camera's view
        extra['in_view_probability'] = torch.sigmoid(output.presence[0, -1]).numpy()
    extra['hand_in_view'] = sample['hand_in_view'][-1].numpy()
    with Path(args.output).open('xb') as stream:
        np.savez_compressed(stream, predicted_xyz=pose, predicted_xyz_cm=pose*clip['world_unit_cm'],
                            target_xyz=sample['target'][-1].numpy(), target_mask=sample['target_mask'][-1].numpy(),
                            world_unit_cm=clip['world_unit_cm'], window_index=window, **extra)
    print(f'Saved {args.output}: [2,21,3] world coordinates and cm')


if __name__ == '__main__':
    main()


In [ ]:
%%writefile /content/gigahands_training/training/export.py
"""Export a trained checkpoint (HandLiteV3 or HandDirect) to ncnn and check the torch-free runtime
against the PyTorch stream on a real clip. Needs `ncnn` and `pnnx`."""
import argparse
import json
from pathlib import Path
import numpy as np
import torch
from hand_tracking.checkpoints import load_checkpoint, stream_for
from hand_tracking.config import architecture_of
from hand_tracking.data import read_clip
from hand_tracking.deploy import export_ncnn
from hand_tracking.direct_export import export_direct
from hand_tracking.direct_runtime import DirectRuntime
from hand_tracking.runtime import LiteV3Runtime

# Architecture -> (exporter, runtime of its export directory)
EXPORTERS = {'litev3': (export_ncnn, LiteV3Runtime), 'direct': (export_direct, DirectRuntime)}


def parse_args(argv=None):
    p = argparse.ArgumentParser(description=__doc__)
    p.add_argument('--checkpoint', required=True)
    p.add_argument('--output', required=True, help='New directory for the graphs and their metadata')
    p.add_argument('--check-input', help='NPZ clip streamed through the runtime and the PyTorch stream')
    p.add_argument('--queries', type=int, default=30, help='Query times compared from the clip (> 0)')
    p.add_argument('--tolerance', type=float, default=2e-3,
                   help='Largest accepted |runtime - PyTorch| in world units (ncnn uses fp16)')
    args = p.parse_args(argv)
    if args.queries < 1 or not args.tolerance > 0:
        p.error('--queries and --tolerance must be positive')
    return args


def compare_on_clip(model, directory, npz, queries):
    """Stream the clip's events through the PyTorch stream and the runtime; largest pose difference."""
    clip = read_clip(npz)
    order = np.argsort(clip['event_arrival'], kind='stable')
    times = clip['query_time'][np.linspace(len(clip['query_time'])//4, len(clip['query_time'])-1, queries).astype(int)]
    pytorch, runtime = stream_for(model), EXPORTERS[architecture_of(model.config)][1](directory)
    worst, pushed, compared = 0., 0, 0
    for time in times:
        while pushed < len(order) and clip['event_arrival'][order[pushed]] <= time:
            i = order[pushed]
            pushed += 1
            for runner in (pytorch, runtime):
                runner.push(clip['event_camera'][i], clip['event_features'][i], clip['event_valid'][i],
                            clip['event_capture'][i], clip['event_arrival'][i])
        if not pushed:
            continue
        expected, actual = pytorch.query(time).numpy(), runtime.query(time)
        if not np.isfinite(expected).all() or not np.isfinite(actual).all():
            raise FloatingPointError(f'Non-finite pose at {time}')
        worst = max(worst, float(np.abs(actual-expected).max()))
        compared += 1
    if not compared:
        raise ValueError(f'No query of {npz} had an arrived event to compare')
    return worst


def main(argv=None):
    args = parse_args(argv)
    torch.set_num_threads(4)
    checkpoint = load_checkpoint(args.checkpoint)
    meta = EXPORTERS[architecture_of(checkpoint.model.config)][0](checkpoint.model, args.output)
    report = dict(output=str(Path(args.output).resolve()), config=meta['config'], graphs=meta['graphs'])
    if args.check_input:
        report['max_abs_difference'] = compare_on_clip(checkpoint.model, args.output, args.check_input, args.queries)
    print(json.dumps(report, indent=2), flush=True)
    if args.check_input and report['max_abs_difference'] > args.tolerance:
        raise SystemExit(f'Runtime differs from PyTorch by more than {args.tolerance}')


if __name__ == '__main__':
    main()


In [ ]:
%%writefile /content/gigahands_training/hand_tracking/__init__.py
"""Event-based multi-camera hand tracking components."""


In [ ]:
%%writefile /content/gigahands_training/hand_tracking/accelerator.py
"""Devices: CPU/CUDA as plain PyTorch, and TPU through PyTorch/XLA (device type 'xla').

XLA traces each step into a graph and compiles it once per distinct tensor shape, so on
XLA: batches keep their fixed padded shapes (no trim_padding, train drops the last partial
batch), host reads (.item(), bool(tensor)) wait for the device and are left to log points,
and each step ends with sync(). torch_xla is imported only when an XLA device is used.
"""
import torch


def is_xla(device):
    return torch.device(device).type == 'xla'


def xla_device(precision='highest'):
    """The TPU device. precision: matmul precision of float32 ('highest' keeps full float32;
    TPUs otherwise may multiply in bfloat16 passes, too coarse for mm-level joints)."""
    import torch_xla
    backends = getattr(torch_xla, 'backends', None)
    if precision and hasattr(backends, 'set_mat_mul_precision'):
        backends.set_mat_mul_precision(precision)
    if hasattr(torch_xla, 'device'):
        return torch_xla.device()
    import torch_xla.core.xla_model as xm
    return xm.xla_device()


def sync(device):
    """End an XLA step: compile (once per shape) and run the traced graph. No-op elsewhere."""
    if not is_xla(device):
        return
    import torch_xla
    if hasattr(torch_xla, 'sync'):
        torch_xla.sync()
    else:
        import torch_xla.core.xla_model as xm
        xm.mark_step()


def device_batches(loader, device):
    """Batches on device: on XLA, a background thread uploads the next batches while the
    TPU runs (MpDeviceLoader, which also ends a step per batch); elsewhere the loader."""
    if not is_xla(device):
        return loader
    from torch_xla.distributed.parallel_loader import MpDeviceLoader
    return MpDeviceLoader(loader, device)


def manual_seed(device, seed):
    if is_xla(device):
        import torch_xla.core.xla_model as xm
        xm.set_rng_state(seed)


def rng_state(device):
    """The XLA random generator's state (dropout), None elsewhere."""
    if not is_xla(device):
        return None
    import torch_xla.core.xla_model as xm
    return xm.get_rng_state()


def to_cpu(value):
    """Tensors (in nested dicts/lists/tuples) moved to CPU, so checkpoints load without torch_xla."""
    if isinstance(value, torch.Tensor):
        return value.detach().cpu()
    if isinstance(value, dict):
        return {key: to_cpu(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return type(value)(to_cpu(item) for item in value)
    return value


In [ ]:
%%writefile /content/gigahands_training/hand_tracking/checkpoints.py
"""Checkpoints: model and input layout together, and the full training state for resuming."""
from dataclasses import asdict, dataclass
import torch
from .accelerator import is_xla, manual_seed, rng_state, to_cpu
from .config import ARCHITECTURES, SamplingConfig, architecture_of, saved_config
from .direct import DirectStream, HandDirect
from .model import HandLiteV3, LiteV3Stream

# Architecture name -> (model, event-by-event stream); config.ARCHITECTURES holds their configs.
MODELS = {'litev3': (HandLiteV3, LiteV3Stream), 'direct': (HandDirect, DirectStream)}
assert set(MODELS) == set(ARCHITECTURES)

INPUT_CONTRACT = ('events [B,E,2,21,14]+valid+camera+capture/arrival seconds+present, query '
                  'times [B,Q] seconds -> pose [B,Q,2,21,3] and expected joint error [B,Q,2,21]; '
                  'or stream_for(model) event by event; XYZ world units')


def build_model(config):
    """The model of a config (config.ARCHITECTURES)."""
    return MODELS[architecture_of(config)][0](config)


def stream_for(model):
    """Event-by-event runner of the model (results equal forward())."""
    return MODELS[architecture_of(model.config)][1](model)


@dataclass
class LoadedCheckpoint:
    model: torch.nn.Module
    sampling: SamplingConfig
    saved: dict


def save_checkpoint(path, value):
    """Write atomically: a crash never leaves a truncated checkpoint at path. Tensors are
    saved on CPU (XLA tensors would need torch_xla to load)."""
    temporary = path.with_suffix('.tmp')
    torch.save(to_cpu(value), temporary)
    temporary.replace(path)


def load_checkpoint(path, device='cpu'):
    saved = torch.load(path, map_location='cpu', weights_only=True)
    if saved.get('architecture') not in MODELS:
        raise ValueError(f"Not a checkpoint of {sorted(MODELS)}: {saved.get('architecture')!r}")
    model = build_model(saved_config(saved['model_config'], saved['architecture'])).to(device)
    model.load_state_dict(saved['model'])
    model.eval()
    return LoadedCheckpoint(model, SamplingConfig(**saved['sampling_config']), saved)


def training_checkpoint(model, sampling, optimizer, scheduler, scaler, epoch, best_val_mpjpe_mm, options):
    """Everything needed to resume after `epoch` (0-based) and to load the model alone."""
    device = next(model.parameters()).device
    cuda = device.type == 'cuda'
    return dict(model=model.state_dict(), architecture=architecture_of(model.config), model_config=asdict(model.config),
                sampling_config=asdict(sampling),
                optimizer=optimizer.state_dict(), scheduler=scheduler.state_dict(), scaler=scaler.state_dict(),
                epoch=epoch, best_val_mpjpe_mm=best_val_mpjpe_mm, rng=torch.get_rng_state(),
                cuda_rng=torch.cuda.get_rng_state_all() if cuda else None, xla_rng=rng_state(device),
                training_options=options,
                torch_version=str(torch.__version__), parameters=sum(p.numel() for p in model.parameters()),
                input_contract=INPUT_CONTRACT)


def restore_training(path, model, optimizer, scheduler, scaler):
    """Load a training_checkpoint into existing objects; returns (next epoch, best val MPJPE)."""
    saved = torch.load(path, map_location='cpu', weights_only=True)
    model.load_state_dict(saved['model'])
    optimizer.load_state_dict(saved['optimizer'])
    scheduler.load_state_dict(saved['scheduler'])
    scaler.load_state_dict(saved['scaler'])
    torch.set_rng_state(saved['rng'])
    device = next(model.parameters()).device
    if device.type == 'cuda' and saved['cuda_rng'] is not None:
        torch.cuda.set_rng_state_all(saved['cuda_rng'])
    if is_xla(device) and saved.get('xla_rng') is not None:
        manual_seed(device, int(saved['xla_rng']))
    return saved['epoch']+1, saved['best_val_mpjpe_mm']


In [ ]:
%%writefile /content/gigahands_training/hand_tracking/config.py
"""Model and input-layout settings. Torch-free, so the deployment runtime can read them. The CLI is
generated from the config's fields, so a new field is automatically a training flag (see add_model_arguments)."""
from dataclasses import dataclass, field, fields
from .constants import NUM_HANDS, FINGERS


# Arrival can trail capture by the pipeline delay; windows carry this much extra history.
ARRIVAL_MARGIN_S = .25


def _option(default, help):
    return field(default=default, metadata=dict(help=help))


@dataclass
class LiteV3Config:
    """HandLiteV3: per camera and joint a line fitted over time to the recent detections, read at the
    query and triangulated (a failed joint anchored from its own last triangulation); a corrector network predicts each joint's offset from that anchor, and its own
    expected error, from the joints' geometry and the recent events' finger tokens
    (hand_tracking.model). No state between queries."""
    dim: int = _option(64, 'Token width of the corrector and event encoder')
    heads: int = _option(4, 'Attention heads')
    blocks: int = _option(2, 'Corrector blocks (cross-attention to event tokens, then over the 42 joints)')
    dropout: float = _option(.1, 'Dropout probability (residual branches)')
    attention_dropout: float = _option(0., 'Dropout probability of attention weights')
    fit_span_s: float = _option(.2, 'Oldest detection the line fit of each camera\'s ray uses (s)')
    # A joint that fails to triangulate now is anchored from its own newest triangulation before the
    # query: the same fit and triangulation at query - k*prior_step_s (k = 1.., within prior_span_s).
    # A stream (and the deployment runtime) also keeps each joint's last triangulation however old
    # (the state) and uses it only when the history search finds none, so its inputs equal training's
    # whenever training could have found one; its age input is capped at max_prior_age_s, the oldest
    # prior point training shows, so a long single-camera stretch stays within the trained range.
    prior_span_s: float = _option(1., 'Oldest reference time searched for a failed joint\'s last triangulation (s)')
    prior_step_s: float = _option(.2, 'Spacing of those reference times (s; fit_span_s: fit windows that do not overlap)')
    # The corrector's joints attend to each camera's slots_per_camera latest events captured
    # within event_span_s, encoded once per event (finger tokens).
    slots_per_camera: int = _option(8, 'Most recent events per camera the corrector attends to')
    event_span_s: float = _option(.5, 'Oldest event capture the corrector attends to (s)')
    # Each joint's own expected error (a confidence the pose output lacks), from its final corrector
    # token with the gradient stopped, so learning it never changes the pose.
    error_estimate: bool = _option(True, 'Also predict each joint\'s own error, log(1 + mm)')
    # Each hand's logit of being inside some camera's view (a hand always exists but can leave all three
    # cameras), from its joints' final corrector tokens (detached, so learning it never changes the pose).
    presence: bool = _option(True, 'Also predict whether each hand is inside some camera\'s view')

    def __post_init__(self):
        if (self.dim < 1 or self.heads < 1 or self.dim % self.heads or self.blocks < 0
                or not 0 <= self.dropout < 1 or not 0 <= self.attention_dropout < 1
                or self.fit_span_s <= 0 or self.slots_per_camera < 1 or self.event_span_s <= 0
                or self.prior_span_s < 0 or self.prior_step_s <= 0):
            raise ValueError('Invalid model configuration')

    @property
    def prior_steps(self):
        """Reference times searched for a failed joint's last triangulation (0: none)."""
        return int(round(self.prior_span_s/self.prior_step_s))

    @property
    def max_prior_age_s(self):
        """Oldest prior point training can show (history search's last reference plus its fit window)."""
        return self.prior_steps*self.prior_step_s+self.fit_span_s

    @property
    def context_s(self):
        """Seconds of events a window needs before its first query (the prior search's fits too)."""
        return max(self.fit_span_s+self.prior_steps*self.prior_step_s, self.event_span_s)+ARRIVAL_MARGIN_S

    @property
    def event_tokens(self):
        """Tokens the event encoder makes per event: one per finger of each hand."""
        return NUM_HANDS*FINGERS


@dataclass
class DirectConfig:
    """HandDirect: networks only, joints straight from the events (hand_tracking.direct). Kept as a
    comparison for HandLiteV3: no triangulation, anchor, error or in-view head."""
    dim: int = _option(64, 'Token width')
    heads: int = _option(4, 'Attention heads')
    fusion_blocks: int = _option(2, 'Self-attention blocks over the slot tokens (views and times)')
    blocks: int = _option(2, 'Decoder blocks (cross-attention to the tokens, then over joints)')
    dropout: float = _option(.1, 'Dropout probability (residual branches)')
    attention_dropout: float = _option(0., 'Dropout probability of attention weights')
    # A query uses each camera's slots_per_camera most recent events captured within
    # event_span_s: a fixed-size input whatever the frame rate or number of events.
    slots_per_camera: int = _option(8, 'Most recent events per camera a query uses')
    event_span_s: float = _option(.5, 'Oldest event capture a query uses (s)')

    def __post_init__(self):
        if (self.dim < 1 or self.heads < 1 or self.dim % self.heads or self.blocks < 1 or self.fusion_blocks < 0
                or not 0 <= self.dropout < 1 or not 0 <= self.attention_dropout < 1 or self.slots_per_camera < 1
                or self.event_span_s <= 0):
            raise ValueError('Invalid model configuration')

    @property
    def context_s(self):
        """Seconds of events a window needs before its first query."""
        return self.event_span_s+ARRIVAL_MARGIN_S

    @property
    def event_tokens(self):
        """Tokens the encoder makes per event: one per finger of each hand."""
        return NUM_HANDS*FINGERS


# Checkpoint/export architecture names; HandLiteV3 is the default, HandDirect a networks-only comparison.
ARCHITECTURES = {'litev3': LiteV3Config, 'direct': DirectConfig}
DEFAULT_ARCHITECTURE = 'litev3'


def architecture_of(config):
    """The ARCHITECTURES name of a config."""
    return next(name for name, cls in ARCHITECTURES.items() if isinstance(config, cls))


def saved_config(values, architecture=DEFAULT_ARCHITECTURE):
    """Config of a saved checkpoint or export."""
    if architecture not in ARCHITECTURES:
        raise ValueError(f'Unknown architecture {architecture!r}; choose from {sorted(ARCHITECTURES)}')
    return ARCHITECTURES[architecture](**values)


def _flag(name, default):
    """error_estimate (default True) -> --no-error-estimate; others -> --name-with-dashes."""
    if default is True:
        return '--no-'+name.replace('_', '-'), 'no_'+name
    return '--'+name.replace('_', '-'), name


def add_model_arguments(parser, config_class=LiteV3Config):
    """One flag per config field, with the field's default and help."""
    group = parser.add_argument_group('model')
    for item in fields(config_class):
        flag, dest = _flag(item.name, item.default)
        if isinstance(item.default, bool):
            text = item.metadata['help']
            group.add_argument(flag, dest=dest, action='store_true', help=f'Disable: {text}' if item.default else text)
        else:
            group.add_argument(flag, dest=dest, type=type(item.default), default=item.default, help=item.metadata['help'])


def model_config_from_args(args, config_class=LiteV3Config):
    values = {}
    for item in fields(config_class):
        _, dest = _flag(item.name, item.default)
        value = getattr(args, dest)
        values[item.name] = (not value) if item.default is True else value
    return config_class(**values)


@dataclass
class SamplingConfig:
    """Input layout settings stored alongside, rather than inside, model weights.
    target_frame: coordinates of the pose targets, 'rig' (the nominal rig as actually placed;
    rig-wide errors no camera reveals removed, see data.rig_frame) or 'world'. mask_out_of_view: a hand
    outside all three cameras (data.hands_in_view) has no position targets; the model learns it is out of view."""
    max_events: int = 64
    target_frame: str = 'rig'
    mask_out_of_view: bool = True

    def __post_init__(self):
        if self.max_events < 1:
            raise ValueError('max_events must be positive')
        if self.target_frame not in ('rig', 'world'):
            raise ValueError("target_frame must be 'rig' or 'world'")


In [ ]:
%%writefile /content/gigahands_training/hand_tracking/constants.py
"""Input layout, units and tolerances shared by data, model and stream."""

NUM_CAMERAS = 3
NUM_HANDS = 2
NUM_JOINTS = 21
# One camera frame gives one token per joint of both hands.
HAND_JOINTS = NUM_HANDS*NUM_JOINTS
# Joint order: wrist 0, then four joints per finger (thumb 1-4, ..., little 17-20).
# A finger group is the wrist and that finger's joints.
FINGERS = 5
FINGER_JOINTS = tuple((0,)+tuple(range(1+4*f, 5+4*f)) for f in range(FINGERS))

# Exported per-joint feature channels [...,FEATURE_CHANNELS]. Channels 8-9 (capture and
# arrival relative to the export query) and 11-13 (ids) are not model inputs: the model
# uses the event's own capture/arrival times instead.
FEATURE_CHANNELS = 14
MODEL_CHANNELS = 11
UV = slice(0, 2)               # normalised pixel coordinates in [0,1]
RAY_ORIGIN = slice(2, 5)       # nominal camera centre, world units
RAY_DIRECTION = slice(5, 8)    # nominal ray direction, world axes
DELAY = slice(10, 11)          # capture->arrival delay, seconds

# Learned features express times, ages and delays in this unit so they are O(1).
TIME_UNIT_S = .1
# Ray-to-point miss vectors and ray shifts (world units / unit directions) are scaled by this,
# and angular residuals (radians) divided by RESIDUAL_UNIT (about 3 px of detector noise), to O(1).
MISS_SCALE = 10.
RESIDUAL_UNIT = .01
# Per-camera calibration (the data's auxiliary target, unused by this model): rotation vector (3) and centre shift (3).
CALIBRATION_PARAMS = 6

# An event is available to a query when it arrived by the query time, up to float32
# rounding of times relative to the batch origin.
ARRIVAL_TOLERANCE_S = 1e-6
# The triangulation input's line fit needs samples spread over at least this time std for a
# slope (otherwise their mean), and two rays at least this far from parallel: sin^2 of the
# widest angle between them; 2e-3 is about 2.6 degrees.
MIN_TIME_STD_S = .005
MIN_RAY_SIN2 = 2e-3
# Additive attention bias of an excluded key; finite so fp16 runtimes stay NaN-free.
MASK_OFF = -1e4
# Streams keep events this long beyond the model's context before pruning.
STREAM_KEEP_MARGIN_S = .5


def check_event(camera, features_shape, valid_shape):
    """Validate one camera frame for a stream; returns the camera index."""
    if int(camera) != camera or not 0 <= int(camera) < NUM_CAMERAS:
        raise ValueError(f'camera must be an integer in [0, {NUM_CAMERAS}), got {camera!r}')
    if tuple(features_shape) != (NUM_HANDS, NUM_JOINTS, FEATURE_CHANNELS) or tuple(valid_shape) != (NUM_HANDS, NUM_JOINTS):
        raise ValueError(f'Expected features [2,21,14] and valid [2,21], got {tuple(features_shape)} and {tuple(valid_shape)}')
    return int(camera)


In [ ]:
%%writefile /content/gigahands_training/hand_tracking/contracts.py
"""Tensor contracts shared by loaders, model, and streaming inference.

Model times are float32 seconds relative to a nearby origin. Stream storage keeps
absolute timestamps in float64. Shapes below include the batch dimension.
"""
from typing import NamedTuple, Optional, TypedDict
from torch import Tensor

MODEL_INPUT_KEYS = ('event_features', 'event_valid', 'event_camera', 'event_capture',
                    'event_arrival', 'event_present', 'query_times')


# Supervision and unit conversion; never model inputs.
TARGET_KEYS = ('target', 'target_world', 'target_mask', 'hand_in_view', 'calibration_target', 'world_unit_cm')


def model_inputs(batch):
    return tuple(batch[key] for key in MODEL_INPUT_KEYS)


class WindowSample(TypedDict):
    """One training/evaluation window (data.make_sample); a DataLoader adds a batch dim.
    Times are float32 seconds relative to the window's last query."""
    event_features: Tensor      # [E,2,21,14], invalid joints zeroed
    event_valid: Tensor         # [E,2,21]
    event_camera: Tensor        # [E] long
    event_present: Tensor       # [E], False = padding
    event_capture: Tensor       # [E]
    event_arrival: Tensor       # [E]
    query_times: Tensor         # [Q]
    target: Tensor              # [Q,2,21,3] world units, in the sampling target_frame (rig or world)
    target_world: Tensor        # [Q,2,21,3] the same in the simulator's true world frame
    target_mask: Tensor         # [Q,2,21] (off for a hand out of all cameras' view, SamplingConfig.mask_out_of_view)
    hand_in_view: Tensor        # [Q,2] whether each hand is inside some camera's view (data.hands_in_view)
    calibration_target: Tensor  # [3,6] in the target frame, NaN when unknown (not used by this model)
    world_unit_cm: Tensor       # []


WINDOW_SAMPLE_KEYS = tuple(WindowSample.__annotations__)
assert set(WINDOW_SAMPLE_KEYS) == set(MODEL_INPUT_KEYS+TARGET_KEYS)


class EventBatch(TypedDict):
    raw: Tensor       # [B,E,2,21,MODEL_CHANNELS], invalid joints zeroed
    valid: Tensor     # [B,E,2,21]
    camera: Tensor    # [B,E]
    capture: Tensor   # [B,E], relative seconds
    arrival: Tensor   # [B,E], relative seconds
    present: Tensor   # [B,E], excludes padding and stale captures


class RawEvents(TypedDict):
    features: Tensor
    valid: Tensor
    camera: Tensor
    capture: Tensor   # stream: [E], absolute float64 seconds
    arrival: Tensor


class ModelOutput(NamedTuple):
    """model(..., return_details=True): calibration is None (no camera calibration head);
    accepted marks the queries with any usable frame or slot; stages is None (one stage);
    error each joint's own expected error, log(1 + mm) (None without the error head); anchored
    how each joint was anchored (model.ANCHOR_KINDS index) and anchor_xyz its anchor; presence each
    hand's logit of being inside some camera's view (None without the presence head)."""
    pose: Tensor                             # [B,Q,2,21,3] world units
    calibration: Optional[Tensor]            # None
    accepted: Tensor                         # [B,Q]
    stages: Optional[Tensor] = None          # None
    error: Optional[Tensor] = None           # [B,Q,2,21] expected joint error, log(1 + mm)
    anchored: Optional[Tensor] = None        # [B,Q,2,21] 0 none, 1 triangulated, 2 one ray at the prior point's depth, 3 the prior point
    anchor_xyz: Optional[Tensor] = None      # [B,Q,2,21,3] world units, the anchor before the correction
    presence: Optional[Tensor] = None        # [B,Q,2] logit: the hand is inside some camera's view


In [ ]:
%%writefile /content/gigahands_training/hand_tracking/data.py
"""Event windows for the time-based model, read from simulator/GigaHands NPZ exports.
Clip-local streaming avoids repeatedly decompressing the same NPZ per window. Samples
follow contracts.WindowSample."""
import json
import os
from pathlib import Path
import numpy as np
import torch
from torch.utils.data import IterableDataset, get_worker_info
from .config import LiteV3Config, SamplingConfig
from .constants import NUM_CAMERAS, NUM_HANDS, NUM_JOINTS, FEATURE_CHANNELS, CALIBRATION_PARAMS, UV, RAY_ORIGIN, \
    RAY_DIRECTION
from .contracts import WindowSample

SPLITS = ('train', 'val', 'test')
# NPZ arrays a clip keeps: model inputs, targets and window indexing.
CLIP_ARRAYS = ('features', 'input_mask', 'target_xyz', 'target_mask', 'query_time', 'window_starts')
# Frame identity and model-visible (reported) timing, used only to recover events.
TIMING_ARRAYS = ('selected_frame', 'capture_time', 'arrival_time')
CALIBRATION_ARRAYS = ('actual_rotations', 'nominal_rotations', 'actual_origins', 'nominal_origins')
# Target frames: 'rig' removes each clip's rig-wide error, which no camera observation
# reveals; 'world' keeps the simulator's true world coordinates.
TARGET_FRAMES = ('rig', 'world')
# Prior scale of the per-camera corrections the rig frame keeps small (the calibrator's).
RIG_FRAME_ROTATION_STD_DEG = 5.
RIG_FRAME_POSITION_STD = .1
# read_clip's arrays that make_sample uses; the cache (cached_clip) keeps only these.
SAMPLE_ARRAYS = ('event_features', 'event_valid', 'event_camera', 'event_capture', 'event_arrival', 'target_xyz',
                 'target_mask', 'query_time', 'window_starts', 'calibration_target', 'rig_target',
                 'rig_calibration_target', 'hand_in_view')
# bump when read_clip's output changes (2: undetected joints' features zeroed; 3: hand_in_view)
CACHE_VERSION = 3
# A hand is in view when at least this share of its joints with a target project inside some
# camera's image (hands_in_view); otherwise it is outside all three cameras.
IN_VIEW_FRACTION = .5


def read_manifest(root):
    """manifest.jsonl rows, checked for valid paths and no participant/source train-val leakage."""
    root = Path(root).resolve()
    lines = (root/'manifest.jsonl').read_text(encoding='utf-8').splitlines()
    rows = [json.loads(line) for line in lines if line.strip()]
    people = {split: set() for split in SPLITS}
    sources = {split: set() for split in SPLITS}
    files = set()
    for row in rows:
        split = row.get('split')
        if split not in people:
            raise ValueError('Manifest requires train/val/test records without errors')
        path = (root/row['file']).resolve()
        if not path.is_relative_to(root) or not path.is_file():
            raise ValueError('Invalid dataset path')
        if row['file'] in files:
            raise ValueError('Duplicate output path')
        files.add(row['file'])
        people[split].add(row['participant'])
        sources[split].add(row['source'])
    if not people['train'] or not people['val']:
        raise ValueError('Both train and val must be present')
    for i, first in enumerate(SPLITS):
        for second in SPLITS[i+1:]:
            if people[first] & people[second] or sources[first] & sources[second]:
                raise ValueError('Participant or source leakage between splits')
    return rows


def rotation_log(matrix):
    """Rotation matrix -> rotation vector (angles well below 180 degrees)."""
    angle = np.arccos(np.clip((np.trace(matrix)-1)/2, -1, 1))
    axis = np.array([matrix[2,1]-matrix[1,2], matrix[0,2]-matrix[2,0], matrix[1,0]-matrix[0,1]])
    return axis*(angle/(2*np.sin(angle)) if angle > 1e-8 else .5)


def rotation_exp(vector):
    """Rotation vector [3] -> matrix (Rodrigues)."""
    angle = np.linalg.norm(vector)
    if angle < 1e-12:
        return np.eye(3)
    k = np.cross(np.eye(3), vector/angle)
    return np.eye(3)+np.sin(angle)*k+(1-np.cos(angle))*k@k


def rig_frame(data, steps=10, tolerance=1e-10):
    """Similarity (rotation [3,3], shift [3], scale) from the true world to the clip's rig
    frame, or None without the simulator rig (real captures).

    Rays only reveal the cameras relative to each other: rotating, moving or scaling the
    whole rig with the hands changes no observation. The rig frame is the true world moved
    by the similarity that leaves the smallest per-camera corrections from the nominal rig
    (rotation / RIG_FRAME_ROTATION_STD_DEG and shift / RIG_FRAME_POSITION_STD, least
    squares, Gauss-Newton): the nominal rig as it was actually placed. Hands keep their
    position around the nominal rig's origin; only the unobservable rig-wide error goes.
    """
    if not all(key in data for key in CALIBRATION_ARRAYS):
        return None
    actual, nominal, actual_origin, nominal_origin = (np.asarray(data[key], np.float64) for key in CALIBRATION_ARRAYS)
    corrections = [a.T@n for a, n in zip(actual, nominal)]                  # calibration_target's rotations
    scales = np.r_[[np.deg2rad(RIG_FRAME_ROTATION_STD_DEG)]*3, [RIG_FRAME_POSITION_STD]*3]

    def residuals(x):
        rotation, shift, scale = rotation_exp(x[:3]), x[3:6], np.exp(x[6])
        rows = [np.r_[rotation_log(rotation@c), scale*rotation@o+shift-n]/scales
                for c, o, n in zip(corrections, actual_origin, nominal_origin)]
        return np.concatenate(rows)

    x = np.zeros(7)
    for _ in range(steps):
        r = residuals(x)
        jacobian = np.stack([(residuals(x+e)-residuals(x-e))/2e-6 for e in np.eye(7)*1e-6], -1)
        step = np.linalg.lstsq(jacobian, r, rcond=None)[0]
        x = x-step
        if np.abs(step).max() < tolerance:
            break
    return rotation_exp(x[:3]), x[3:6], float(np.exp(x[6]))


def calibration_target(data):
    """Simulator's true correction of the nominal rays [3,6], NaN when unknown (real captures).

    Same convention as geometry.apply_calibration: world-axis rotation about the camera
    centre (R_actual^T R_nominal, since rays are pixel @ R) and centre shift.
    Supervision only; never a model input.
    """
    if not all(key in data for key in CALIBRATION_ARRAYS):
        return np.full((NUM_CAMERAS, CALIBRATION_PARAMS), np.nan, np.float32)
    rows = [np.concatenate([rotation_log(actual.T@nominal), actual_origin-nominal_origin])
            for actual, nominal, actual_origin, nominal_origin in zip(*(data[key] for key in CALIBRATION_ARRAYS))]
    return np.asarray(rows, np.float32)


def to_rig_frame(target, calibration, data, frame):
    """target [...,3] and calibration target [3,6] in the rig frame (see rig_frame): the hands
    moved by the similarity, the corrections those that turn the nominal rays into the moved
    true rays. Unchanged for real captures, which have no simulator rig."""
    if frame is None:
        return target, calibration
    rotation, shift, scale = frame
    nominal_origin = np.asarray(data['nominal_origins'], np.float64)
    actual_origin = np.asarray(data['actual_origins'], np.float64)
    moved = np.asarray([np.r_[rotation_log(rotation@rotation_exp(row[:3])), scale*rotation@origin+shift-nominal]
                        for row, origin, nominal in zip(calibration.astype(np.float64), actual_origin, nominal_origin)])
    return (scale*target@rotation.T+shift).astype(np.float32), moved.astype(np.float32)


def pixel_model(features, mask, nominal_rotations):
    """Each camera's pinhole pixel mapping fitted from the clip's detections: u = a_u x + b_u and
    v = a_v y + b_v, (x, y) the ray's normalised camera coordinates (camera frame = the nominal
    rotation applied to the ray direction, since rays = pixel @ R). features [T,3,2,21,14], mask
    [T,3,2,21] -> coefficients [3,2,2] (camera; u, v; slope, offset) and the fit's RMS residual
    [3] (image units, [0,1]); NaN for a camera with fewer than 10 usable detections."""
    rotations = np.asarray(nominal_rotations, np.float64)
    coefficients, rms = np.full((NUM_CAMERAS, 2, 2), np.nan), np.full(NUM_CAMERAS, np.nan)
    for camera in range(NUM_CAMERAS):
        seen = features[:, camera][mask[:, camera].astype(bool)].astype(np.float64)       # [n,14]
        c = seen[:, RAY_DIRECTION]@rotations[camera].T                                      # camera frame
        front = c[:, 2] > 1e-9
        if front.sum() < 10:
            continue
        xy, uv = c[front, :2]/c[front, 2:], seen[front][:, UV]
        residuals = []
        for axis in range(2):
            design = np.stack((xy[:, axis], np.ones(len(xy))), -1)
            coefficients[camera, axis] = np.linalg.lstsq(design, uv[:, axis], rcond=None)[0]
            residuals.append(uv[:, axis]-design@coefficients[camera, axis])
        rms[camera] = float(np.sqrt(np.mean(np.square(residuals))))
    return coefficients, rms


def hands_in_view(clip, data, fraction=IN_VIEW_FRACTION):
    """Per query time and hand [T,2]: whether the hand is in some camera's view, i.e. at least
    `fraction` of its joints with a target project inside that camera's image ([0,1]^2, in front
    of it) through the true camera pose (actual rotation and origin, world-frame targets) and the
    pixel mapping fitted from the detections (pixel_model). A hand outside all three cameras still
    exists but nothing can see it. All True without the simulator's cameras (real captures) or a
    usable fit; a camera without a fit (it detected nothing in the clip) vouches for no hand."""
    everywhere = np.ones((len(clip['target_xyz']), NUM_HANDS), bool)
    if not all(key in data for key in CALIBRATION_ARRAYS):
        return everywhere
    coefficients, _ = pixel_model(clip['features'], clip['input_mask'], data['nominal_rotations'])
    if np.isnan(coefficients).all():
        return everywhere
    rotations, origins = np.asarray(data['actual_rotations'], np.float64), np.asarray(data['actual_origins'], np.float64)
    target, valid = clip['target_xyz'].astype(np.float64), clip['target_mask'].astype(bool)
    in_view = np.zeros_like(everywhere)
    for camera in range(NUM_CAMERAS):
        if np.isnan(coefficients[camera]).any():
            continue
        c = (target-origins[camera])@rotations[camera].T                                    # [T,2,21,3]
        front = c[..., 2] > 1e-9
        depth = np.where(front, c[..., 2], 1.)
        u = coefficients[camera, 0, 0]*c[..., 0]/depth+coefficients[camera, 0, 1]
        v = coefficients[camera, 1, 0]*c[..., 1]/depth+coefficients[camera, 1, 1]
        inside = front & (u >= 0) & (u <= 1) & (v >= 0) & (v <= 1) & valid
        in_view |= inside.sum(-1) >= fraction*np.maximum(valid.sum(-1), 1)
    return in_view


def read_clip(path):
    with np.load(path, allow_pickle=False) as data:
        clip = {key: data[key].copy() for key in CLIP_ARRAYS}
        clip['calibration_target'] = calibration_target(data)
        frame = rig_frame(data)
        clip['rig_target'], clip['rig_calibration_target'] = to_rig_frame(clip['target_xyz'], clip['calibration_target'],
                                                                           data, frame)
        clip['hand_in_view'] = hands_in_view(clip, data)
        timing = {key: data[key].copy() for key in TIMING_ARRAYS}
        meta = json.loads(str(data['metadata']))
    clip['window'] = int(meta['config']['window'])
    clip['world_unit_cm'] = float(meta['units']['world_unit_cm'])
    clip.update(clip_events(clip, timing))
    return clip


def cached_clip(root, file, cache=None):
    """make_sample's arrays of read_clip(root/file). With a cache folder, the first read also
    writes them uncompressed to cache/file and later reads load that instead, skipping the
    NPZ decompression and the rig-frame fit; an entry is rebuilt when its source changes."""
    path = Path(root)/file
    if cache:
        entry = Path(cache)/file
        source = path.stat()
        stamp = np.array([CACHE_VERSION, source.st_size, source.st_mtime_ns])
        try:
            with np.load(entry, allow_pickle=False) as data:
                if np.array_equal(data['stamp'], stamp):
                    clip = {key: data[key] for key in data.files if key != 'stamp'}
                    clip['window'], clip['world_unit_cm'] = int(clip['window']), float(clip['world_unit_cm'])
                    return clip
        except (OSError, KeyError, ValueError):   # missing, stale format or a partial write
            pass
    clip = read_clip(path)
    clip = {key: clip[key] for key in SAMPLE_ARRAYS + ('window', 'world_unit_cm') if key in clip}
    if cache:
        entry.parent.mkdir(parents=True, exist_ok=True)
        partial = entry.with_name(f'{entry.name}.{os.getpid()}.tmp')
        with partial.open('wb') as f:
            np.savez(f, stamp=stamp, **clip)
        os.replace(partial, entry)   # readers never see a half-written entry
    return clip


def clip_events(clip, timing):
    """Camera frames (events) recovered from the query-grid export, sorted by arrival.

    Each query holds every camera's selected (latest captured, arrived) frame; a frame is
    an event at the first query that selects it, including frames in which no joint was
    detected (they still replace that camera's older rays, as live). Stale out-of-order
    arrivals never get selected, as a stream drops them. A frame superseded before any
    query (two arrivals of one camera between queries) is not recoverable from this format.
    """
    selected = timing['selected_frame']
    new = selected >= 0
    new[1:] &= selected[1:] != selected[:-1]
    frame, camera = np.nonzero(new)
    arrival = timing['arrival_time'][frame, camera].astype(np.float64)
    capture = timing['capture_time'][frame, camera].astype(np.float64)
    # Ties keep (query, camera) order; the model treats input order as arrival order.
    order = np.argsort(arrival, kind='stable')
    # Undetected joints' features are zeroed once here, not in every window (make_sample).
    features = clip['features'][frame, camera][order].astype(np.float32, copy=False)
    valid = clip['input_mask'][frame, camera][order].astype(bool, copy=False)
    features[~valid] = 0
    return dict(event_features=features, event_valid=valid,
                event_camera=camera[order].astype(np.int64), event_capture=capture[order],
                event_arrival=arrival[order])


def make_sample(clip, window_index, context_s=None, max_events=SamplingConfig.max_events,
                target_frame=SamplingConfig.target_frame, queries=0,
                mask_out_of_view=SamplingConfig.mask_out_of_view) -> WindowSample:
    """Time-based window: the events arrived by the window's last query and captured at most
    context_s before its first query, and the window's query times with their targets.

    Times are seconds relative to the last query. Events are padded to max_events
    (event_present False); if more arrive, the oldest are dropped. target is in target_frame ('rig' or 'world', see
    rig_frame; a clip without the simulator rig has no rig-wide error to remove);
    target_world always in the world frame. queries > 0 keeps only the window's last
    queries (the evaluated last query included) and the events their history needs.
    hand_in_view [Q,2] says whether each hand is inside some camera's view (hands_in_view, from
    the true geometry); with mask_out_of_view a hand outside all three cameras has no position
    targets (it exists, but nothing can see where): the model learns that it is out of view.
    A hand in view but not detected keeps its targets.
    """
    if target_frame not in TARGET_FRAMES:
        raise ValueError(f'target_frame must be one of {TARGET_FRAMES}')
    if context_s is None:
        context_s = LiteV3Config().context_s
    end = int(clip['window_starts'][window_index])+clip['window']
    if queries < 0 or queries > clip['window']:
        raise ValueError(f"queries must be in [0, {clip['window']}]")
    start = end-queries if queries else end-clip['window']
    query = clip['query_time'][start:end].astype(np.float64)
    last, first = query[-1], query[0]-context_s
    arrival, capture = clip['event_arrival'], clip['event_capture']
    # Events are sorted by arrival and none is captured after it arrives, so every candidate
    # arrived between the history start and the last query: a slice found by binary search
    # instead of a scan of the whole clip (checked once per clip; otherwise the whole clip).
    if '_causal' not in clip:
        clip['_causal'] = bool(np.all(capture <= arrival+1e-9))
    lo = np.searchsorted(arrival, first, 'left') if clip['_causal'] else 0
    hi = np.searchsorted(arrival, last+1e-9, 'right')
    chosen = (lo+np.nonzero(capture[lo:hi] >= first)[0])[-max_events:]
    n = len(chosen)
    features = np.zeros((max_events, NUM_HANDS, NUM_JOINTS, FEATURE_CHANNELS), np.float32)
    valid = np.zeros((max_events, NUM_HANDS, NUM_JOINTS), bool)
    camera = np.zeros(max_events, np.int64)
    present = np.zeros(max_events, bool)
    event_capture = np.zeros(max_events, np.float32)
    event_arrival = np.zeros(max_events, np.float32)
    features[:n] = clip['event_features'][chosen]          # undetected joints already zeroed (clip_events)
    valid[:n] = clip['event_valid'][chosen]
    camera[:n] = clip['event_camera'][chosen]
    present[:n] = True
    event_capture[:n] = capture[chosen]-last
    event_arrival[:n] = arrival[chosen]-last
    if n and event_arrival[:n].max() > 1e-6:
        raise ValueError('Future arrival in historical input')
    if target_frame == 'rig' and 'rig_target' in clip:
        target, calibration = clip['rig_target'][start:end], clip['rig_calibration_target']
    else:
        target, calibration = clip['target_xyz'][start:end], clip['calibration_target']
    target_mask = clip['target_mask'][start:end].astype(bool)
    in_view = (clip['hand_in_view'][start:end].astype(bool) if 'hand_in_view' in clip
               else np.ones((end-start, NUM_HANDS), bool))
    if mask_out_of_view:
        target_mask = target_mask & in_view[..., None]
    as_float = lambda value: torch.from_numpy(np.array(value, dtype=np.float32))   # one copy
    return WindowSample(event_features=torch.from_numpy(features), event_valid=torch.from_numpy(valid),
                        event_camera=torch.from_numpy(camera), event_present=torch.from_numpy(present),
                        event_capture=torch.from_numpy(event_capture), event_arrival=torch.from_numpy(event_arrival),
                        query_times=as_float(query-last), target=as_float(target),
                        target_world=as_float(clip['target_xyz'][start:end]),
                        target_mask=torch.from_numpy(target_mask), hand_in_view=torch.from_numpy(in_view.copy()),
                        calibration_target=as_float(calibration),
                        world_unit_cm=torch.tensor(clip['world_unit_cm'], dtype=torch.float32))


def trim_padding(batch):
    """Batch with the event axis cut to its largest present count. make_sample puts padding
    after the events and the model ignores it, so outputs are unchanged; the event axis is
    then as long as the batch needs (max_events is a cap, not a cost)."""
    present = batch['event_present']
    if present.ndim != 2:
        return batch
    used = max(int(present.sum(1).max()), 1)
    if used == present.shape[1] or present[:, used:].any():
        return batch
    return {key: value[:, :used] if key.startswith('event_') else value for key, value in batch.items()}


class HandWindows(IterableDataset):
    """Windows of one split. Train: shuffled clips and random windows, reseeded per epoch
    (set .epoch). Val: fixed, evenly spaced windows. No augmentation: the camera rig is
    fixed, so its world coordinates are worth learning."""

    def __init__(self, root, split, windows_per_clip=16, seed=42, max_clips=0, context_s=None, max_events=SamplingConfig.max_events, target_frame=SamplingConfig.target_frame,
                 queries=0, cache=None, mask_out_of_view=SamplingConfig.mask_out_of_view):
        super().__init__()
        if context_s is None:
            context_s = LiteV3Config().context_s
        if (split not in SPLITS or windows_per_clip < 0 or max_clips < 0 or context_s < 0 or max_events < 1
                or target_frame not in TARGET_FRAMES or queries < 0):
            raise ValueError('Invalid sampling settings')
        self.root = Path(root)
        self.rows = [row for row in read_manifest(root) if row['split'] == split and row['windows'] > 0]
        if max_clips:
            self.rows = self.rows[:max_clips]
        if not self.rows:
            raise ValueError('No usable clips')
        self.split, self.windows_per_clip, self.seed, self.epoch = split, windows_per_clip, seed, 0
        self.context_s, self.max_events, self.target_frame, self.queries = context_s, max_events, target_frame, queries
        self.cache = cache   # folder for cached_clip; None reads the NPZ every time
        self.mask_out_of_view = mask_out_of_view   # make_sample: no position targets for hands outside all cameras

    def _windows(self, count):
        return min(count, self.windows_per_clip) if self.windows_per_clip else count

    def __len__(self):
        return sum(self._windows(row['windows']) for row in self.rows)

    def __iter__(self):
        training = self.split == 'train'
        indices = np.arange(len(self.rows))
        if training:
            np.random.default_rng(self.seed+self.epoch).shuffle(indices)
        worker = get_worker_info()
        if worker:
            indices = indices[worker.id::worker.num_workers]
        for index in indices:
            row = self.rows[index]
            clip = cached_clip(self.root, row['file'], self.cache)
            count = len(clip['window_starts'])
            if count != row['windows']:
                raise ValueError('Manifest window count differs from NPZ')
            if training:
                rng = np.random.default_rng(np.random.SeedSequence([self.seed, self.epoch, int(index)]))
                chosen = rng.choice(count, self._windows(count), replace=False)
            else:
                chosen = np.linspace(0, count-1, self._windows(count), dtype=int)
            for window in chosen:
                sample = make_sample(clip, window, self.context_s, self.max_events, self.target_frame,
                                     self.queries, self.mask_out_of_view)
                # Event-less windows cannot constrain an observation-based estimator. A window whose
                # hands are both out of view at the final query still teaches that they are.
                if not sample['event_present'].any():
                    continue
                yield sample


In [ ]:
%%writefile /content/gigahands_training/hand_tracking/deploy.py
"""Export HandLiteV3's networks to ncnn (converted by pnnx) for hand_tracking.runtime. Shapes have
no batch axis in ncnn. T = 10 * 3 * slots_per_camera, F = model.LITE3_FEATURES.

    encoder    event features [2, 21*10], camera one-hot [3]                   -> finger tokens [10, dim]
    corrector  joint features [42, F], slot tokens [T, dim], validity 0/1 [T], ages [T]
               -> offsets [42, 3], then with config.error_estimate each joint's expected error log(1 + mm) and
                  with config.presence its hand's in-view logit: [42, 3 to 5]
"""
from dataclasses import asdict
import contextlib
import json
import os
from pathlib import Path
import torch
from .constants import NUM_CAMERAS, NUM_HANDS, NUM_JOINTS, HAND_JOINTS
from .model import EVENT_TOKENS, LITE3_FEATURES, CorrectorGraph
from .networks import DIRECT_JOINT_FEATURES
from .runtime import EXPORT_FORMAT, GRAPHS, META_FILE


def graphs(model):
    """name -> (module, example inputs with a batch of one)."""
    model = model.eval()
    torch.manual_seed(0)
    tokens = model.slots*EVENT_TOKENS
    return dict(encoder=(model.encoder, (torch.randn(1, NUM_HANDS, NUM_JOINTS*DIRECT_JOINT_FEATURES),
                                         torch.eye(NUM_CAMERAS)[:1])),
                corrector=(CorrectorGraph(model.corrector), (torch.randn(1, HAND_JOINTS, LITE3_FEATURES),
                                                             torch.randn(1, tokens, model.config.dim),
                                                             torch.ones(1, tokens), torch.rand(1, tokens))))


def export_ncnn(model, directory):
    """Write the ncnn graphs and litev3.json (config and graph input shapes) to a new directory;
    returns the metadata."""
    directory = Path(directory)
    directory.mkdir(parents=True)
    exported = graphs(model)
    with torch.no_grad():
        for name in GRAPHS:
            _pnnx(*exported[name], directory, name)
    meta = dict(export_format=EXPORT_FORMAT, config=asdict(model.config), graphs={
        name: dict(inputs=[list(value.shape[1:]) for value in exported[name][1]]) for name in GRAPHS})
    (directory/META_FILE).write_text(json.dumps(meta, indent=2), encoding='utf-8')
    return meta


@contextlib.contextmanager
def _working_directory(path):
    previous = Path.cwd()
    os.chdir(path)
    try:
        yield
    finally:
        os.chdir(previous)


def _pnnx(module, example, directory, name):
    """Convert with pnnx and keep only the ncnn param/bin (pnnx writes next to its input)."""
    import pnnx
    with _working_directory(directory):
        pnnx.export(module, f'{name}.pt', example)
        for path in list(Path('.').glob(f'{name}*'))+list(Path('__pycache__').glob(f'{name}*')):
            if not path.name.endswith(('.ncnn.param', '.ncnn.bin')):
                path.unlink()
        cache = Path('__pycache__')
        if cache.is_dir() and not any(cache.iterdir()):
            cache.rmdir()


In [ ]:
%%writefile /content/gigahands_training/hand_tracking/direct.py
"""HandDirect: camera events to the 3D joints of both hands with networks only.

  per event, once, on arrival (from that frame alone)
    features  u,v, ray origin and direction, capture->arrival delay, detected   [2,21,10]
    encoder   one token per finger of each hand                                [10, dim]
  per query at time t
    slots     each camera's slots_per_camera latest events captured within event_span_s
    fusion    self-attention over the slots' tokens, each told its age (t - capture)
    decoder   42 joint queries attend to the fused tokens -> xyz                  [42, 3]

No triangulation, anchor or camera correction: the networks learn how views and times
combine. forward() computes exactly what DirectStream computes event by event. Kept as the
networks-only comparison for HandLiteV3 (hand_tracking.model), whose event encoder it shares.
"""
import torch
from torch import nn
from .config import DirectConfig
from .constants import NUM_CAMERAS, NUM_HANDS, NUM_JOINTS, HAND_JOINTS, FEATURE_CHANNELS, TIME_UNIT_S, MASK_OFF
from .contracts import ModelOutput
from .events import _gather, make_events, relative_events, select_slots
from .layers import Block, EventEncoder
from .networks import DIRECT_JOINT_FEATURES, DirectEncoded, camera_one_hot, direct_features, repeat_each
from .stream import EventStream


class Fusion(nn.Module):
    """tokens [N,T,dim], validity [N,T] (1/0), ages [N,T] (TIME_UNIT_S) -> fused tokens [N,T,dim].
    Each token first adds an embedding of its age; padding tokens stay zero and are masked."""

    def __init__(self, dim, heads, blocks, dropout, attention_dropout):
        super().__init__()
        self.age = nn.Sequential(nn.Linear(1, 16), nn.GELU(), nn.Linear(16, dim))
        self.blocks = nn.ModuleList([Block(dim, heads, dropout, cross=False, attention_dropout=attention_dropout)
                                     for _ in range(blocks)])

    def forward(self, tokens, valid, ages):
        x = (tokens+self.age(ages[..., None]))*valid[..., None]
        bias = ((valid-1.)*-MASK_OFF)[:, None, None]                                         # [N,1,1,T]
        for block in self.blocks:
            x = block(x, self_bias=bias)*valid[..., None]
        return x


class Decoder(nn.Module):
    """fused tokens [N,T,dim], validity [N,T] (1/0) -> joints [N,42,3]: learned joint queries
    attend to the valid tokens, then to each other."""

    def __init__(self, dim, heads, blocks, dropout, attention_dropout):
        super().__init__()
        self.joints = nn.Parameter(torch.randn(HAND_JOINTS, dim)*.02)
        self.null = nn.Parameter(torch.zeros(1, dim))
        self.blocks = nn.ModuleList([Block(dim, heads, dropout, attention_dropout=attention_dropout)
                                     for _ in range(blocks)])
        self.output = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, dim), nn.GELU(), nn.Linear(dim, 3))

    def forward(self, tokens, valid):
        n, _, d = tokens.shape
        keys = torch.cat((self.null.expand(n, 1, d), tokens), 1)                           # a query with no slot
        bias = torch.cat((torch.zeros_like(valid[:, :1]), (valid-1.)*-MASK_OFF), -1)[:, None, None]
        x = self.joints.expand(n, -1, -1)
        for block in self.blocks:
            x = block(x, keys, bias)
        return self.output(x)


class QueryNetwork(nn.Module):
    """The per-query networks as one graph: fusion then decoder (see lite export)."""

    def __init__(self, fusion, decoder):
        super().__init__()
        self.fusion, self.decoder = fusion, decoder

    def forward(self, tokens, valid, ages):
        return self.decoder(self.fusion(tokens, valid, ages), valid)


class HandDirect(nn.Module):
    def __init__(self, config=None):
        super().__init__()
        self.config = config or DirectConfig()
        c = self.config
        self.encoder = EventEncoder(c.dim, True, DIRECT_JOINT_FEATURES)
        self.query_network = QueryNetwork(Fusion(c.dim, c.heads, c.fusion_blocks, c.dropout, c.attention_dropout),
                                          Decoder(c.dim, c.heads, c.blocks, c.dropout, c.attention_dropout))

    @property
    def slots(self):
        return NUM_CAMERAS*self.config.slots_per_camera

    def encode_events(self, events, targets=None):
        """DirectEncoded of the target events (default: all); each from its own frame only."""
        b, e = events['camera'].shape
        if targets is None:
            targets = torch.arange(e, device=events['camera'].device)
        t = targets.numel()
        features = direct_features(events['raw'][:, targets], events['valid'][:, targets])
        camera = camera_one_hot(events['camera'][:, targets])
        tokens = self.encoder(features.reshape(b*t, NUM_HANDS, -1), camera.reshape(b*t, NUM_CAMERAS))
        return DirectEncoded(tokens.reshape(b, t, *tokens.shape[1:]))

    def decode_queries(self, events, encoded, query):
        """Pose [B,Q,2,21,3], None (no camera correction) and whether a query had any slot [B,Q]."""
        b, q = query.shape
        n, s, per = b*q, self.slots, self.config.event_tokens
        slot, valid = select_slots(events, query, self.config.slots_per_camera, self.config.event_span_s)
        take = lambda value: _gather(value, slot).reshape(n, s, *value.shape[2:])
        valid = valid.reshape(n, s)
        token_valid = repeat_each(valid, per)
        tokens = torch.where(token_valid[..., None], take(encoded.tokens).reshape(n, per*s, -1), 0.)
        ages = torch.where(valid, ((query.reshape(n, 1)-take(events['capture']))/TIME_UNIT_S).clamp_min(0.), 0.)
        pose = self.query_network(tokens, token_valid.to(tokens.dtype), repeat_each(ages, per))
        return pose.float().reshape(b, q, NUM_HANDS, NUM_JOINTS, 3), None, valid.any(-1).reshape(b, q)

    def forward(self, event_features, event_valid, event_camera, event_capture, event_arrival, event_present,
                query_times, return_details=False):
        """Model inputs (contracts.MODEL_INPUT_KEYS). return_details gives ModelOutput with no
        calibration and accepted marking queries with any slot [B,Q]."""
        if event_features.ndim != 5 or event_features.shape[2:] != (NUM_HANDS, NUM_JOINTS, FEATURE_CHANNELS):
            raise ValueError('Expected event features [B,E,2,21,14]')
        if event_valid.shape != event_features.shape[:-1] or query_times.ndim != 2:
            raise ValueError('Invalid event/query dimensions')
        events = make_events(event_features, event_valid, event_camera, event_capture, event_arrival, event_present)
        pose, calibration, rows = self.decode_queries(events, self.encode_events(events), query_times.float())
        return ModelOutput(pose, calibration, rows) if return_details else pose


class DirectStream(EventStream):
    """HandDirect event by event: each event's tokens are computed once; a query selects the
    slots and runs fusion and decoder. Equals forward()."""

    def decode(self, time):
        encoded = self.store.encoded.map(lambda value: value[None])
        query = torch.zeros(1, 1, device=self.device)
        return self.model.decode_queries(relative_events(self.store.raw, time), encoded, query)[0][0, 0]


In [ ]:
%%writefile /content/gigahands_training/hand_tracking/direct_export.py
"""Export HandDirect's two fixed-shape networks to ncnn (converted by pnnx) for direct_runtime.

    encoder  features [2, 21*10], camera one-hot [3]                     -> tokens [10, dim]
    query    tokens [10*3K, dim], token validity 0/1 [10*3K], ages [10*3K] -> joints [42, 3]
"""
from dataclasses import asdict
import json
from pathlib import Path
import torch
from .constants import NUM_CAMERAS, NUM_HANDS, NUM_JOINTS
from .deploy import _pnnx
from .direct_runtime import EXPORT_FORMAT, GRAPHS, META_FILE
from .networks import DIRECT_JOINT_FEATURES


def graphs(model):
    """name -> (module, example inputs with a batch of one)."""
    model = model.eval()
    tokens = model.config.event_tokens*model.slots
    torch.manual_seed(0)
    return dict(encoder=(model.encoder, (torch.randn(1, NUM_HANDS, NUM_JOINTS*DIRECT_JOINT_FEATURES),
                                         torch.eye(NUM_CAMERAS)[:1])),
                query=(model.query_network, (torch.randn(1, tokens, model.config.dim), torch.ones(1, tokens),
                                             torch.rand(1, tokens))))


def export_direct(model, directory):
    """Write the ncnn graphs and direct.json (config and graph input shapes) to a new directory;
    returns the metadata."""
    directory = Path(directory)
    directory.mkdir(parents=True)
    exported = graphs(model)
    with torch.no_grad():
        for name in GRAPHS:
            _pnnx(*exported[name], directory, name)
    meta = dict(architecture='direct', export_format=EXPORT_FORMAT, config=asdict(model.config), graphs={
        name: dict(inputs=[list(value.shape[1:]) for value in exported[name][1]]) for name in GRAPHS})
    (directory/META_FILE).write_text(json.dumps(meta, indent=2), encoding='utf-8')
    return meta


In [ ]:
%%writefile /content/gigahands_training/hand_tracking/direct_runtime.py
"""HandDirect deployment runtime without PyTorch: slot selection around the two exported
networks, run by ncnn. Mirrors hand_tracking.direct.DirectStream.

    runtime = DirectRuntime('exports/direct_model')
    runtime.push(camera, features [2,21,14], valid [2,21], capture_time, arrival_time)
    pose = runtime.query(time)          # [2,21,3] world units

Times are absolute seconds (float64). Needs numpy and `ncnn`.
"""
from dataclasses import dataclass
import json
from pathlib import Path
import numpy as np
from .config import DirectConfig
from .constants import (NUM_CAMERAS, NUM_HANDS, NUM_JOINTS, MODEL_CHANNELS, UV, RAY_ORIGIN, RAY_DIRECTION, DELAY,
                        TIME_UNIT_S, STREAM_KEEP_MARGIN_S, check_event)
from .runtime import NcnnGraphs, select_slot_events

META_FILE = 'direct.json'
EXPORT_FORMAT = 2   # 2: ncnn only (pnnx), graph input shapes in the metadata
GRAPHS = ('encoder', 'query')


def direct_features(raw, valid):
    """Event features [2,21,11] (invalid joints zeroed), valid -> encoder input [2,21,10]."""
    x = np.concatenate((raw[..., UV]*2-1, raw[..., RAY_ORIGIN], raw[..., RAY_DIRECTION], raw[..., DELAY]/TIME_UNIT_S,
                        valid[..., None]), -1)
    return np.where(valid[..., None], x, 0.)


@dataclass
class _Event:
    camera: int
    capture: float
    arrival: float
    tokens: np.ndarray     # [10,dim]


class DirectRuntime:
    def __init__(self, directory, threads=1, fp16=True, graphs=None):
        """graphs: any object with run(name, *arrays), instead of loading ncnn."""
        directory = Path(directory)
        meta = json.loads((directory/META_FILE).read_text(encoding='utf-8'))
        if meta.get('architecture') != 'direct' or meta.get('export_format') != EXPORT_FORMAT:
            raise ValueError(f'{directory} is not a HandDirect export of format {EXPORT_FORMAT}')
        self.config = DirectConfig(**meta['config'])
        self.graphs = graphs if graphs is not None else NcnnGraphs(directory, GRAPHS, threads, fp16)
        self.keep_s = self.config.context_s+STREAM_KEEP_MARGIN_S
        self.reset()

    def reset(self):
        self.events = []
        self.newest_capture = [None]*NUM_CAMERAS
        self.last_arrival = None

    def __len__(self):
        return len(self.events)

    def push(self, camera, features, valid, capture_time, arrival_time):
        """Encode one arrived camera frame; False (ignored) for a capture no newer than that
        camera's latest. Frames must arrive in order; frames without detections count."""
        camera = check_event(camera, np.shape(features), np.shape(valid))
        capture, arrival = float(capture_time), float(arrival_time)
        if self.last_arrival is not None and arrival < self.last_arrival:
            raise ValueError('Events must be pushed in arrival order')
        self.last_arrival = arrival
        if self.newest_capture[camera] is not None and capture <= self.newest_capture[camera]:
            return False
        self.newest_capture[camera] = capture
        valid = np.array(valid, dtype=bool, copy=True)
        raw = np.where(valid[..., None], np.asarray(features, np.float64)[..., :MODEL_CHANNELS], 0.)
        tokens = self.graphs.run('encoder', direct_features(raw, valid).reshape(NUM_HANDS, -1), np.eye(NUM_CAMERAS)[camera])
        self.events.append(_Event(camera, capture, arrival, np.asarray(tokens)))
        while self.events and self.events[0].capture < arrival-self.keep_s:
            self.events.pop(0)
        return True

    def query(self, time):
        """Pose [2,21,3] at time (not before the latest arrival)."""
        time = float(time)
        if self.last_arrival is None:
            raise ValueError('No events yet')
        if time < self.last_arrival:
            raise ValueError('Query before the latest arrival')
        slots = select_slot_events(self.events, time, self.config.slots_per_camera, self.config.event_span_s)
        per, dim = self.config.event_tokens, self.config.dim
        present = np.array([event is not None for event in slots])
        tokens = np.stack([event.tokens if event is not None else np.zeros((per, dim)) for event in slots])
        ages = np.array([max(time-event.capture, 0.)/TIME_UNIT_S if event is not None else 0. for event in slots])
        joints = self.graphs.run('query', tokens.reshape(-1, dim), np.repeat(present, per), np.repeat(ages, per))
        return np.asarray(joints).reshape(NUM_HANDS, NUM_JOINTS, 3)


In [ ]:
%%writefile /content/gigahands_training/hand_tracking/engine.py
"""Epoch runner shared by training and validation (CPU, CUDA or XLA/TPU; see accelerator)."""
import contextlib
import time
import torch
from .accelerator import device_batches, is_xla, sync
from .contracts import model_inputs
from .data import trim_padding
from .model import ANCHOR_KINDS
from .objectives import PoseTotals, WeightedMean, _number, pose_loss, relative_loss


def to_device(batch, device):
    return {key: value.to(device, non_blocking=True) for key, value in batch.items()}


def joint_error_log(pose, batch):
    """Each joint's actual error log(1 + mm) [B,Q,2,21] (detached) and where it has a target."""
    mask = batch['target_mask'].bool()
    mm = batch['world_unit_cm'].reshape(-1, *[1]*(pose.ndim-2))*10
    error = (pose.detach().float()-torch.where(mask[..., None], batch['target'], 0.)).norm(dim=-1)*mm
    return torch.log1p(error), mask.float()


def batch_loss(batch, output, bone_weight, relative_weight=0., error_weight=0., presence_weight=0.):
    """Pose loss in the target frame (the rig frame: the rig-wide error no observation reveals is
    not a target) over every query time, plus relative_weight times the wrist-relative loss (hand
    shape, free of the wrist position error), plus error_weight times the error estimate loss
    (SmoothL1 of output.error against each joint's actual error, log(1 + mm), over joints with a
    target; the estimate's input is detached, so this never changes the pose), plus
    presence_weight times the in-view loss (binary cross-entropy of output.presence against
    hand_in_view, every hand; also on detached tokens). A hand out of all cameras' view has no
    position targets (data.make_sample) and only teaches that it is out of view."""
    loss = pose_loss(output.pose, batch['target'], batch['target_mask'], batch['world_unit_cm'], bone_weight)
    if relative_weight:
        loss = loss+relative_weight*relative_loss(output.pose, batch['target'], batch['target_mask'], batch['world_unit_cm'])
    if error_weight and output.error is not None:
        actual, mask = joint_error_log(output.pose, batch)
        misses = torch.nn.functional.smooth_l1_loss(output.error, actual, beta=.1, reduction='none')
        loss = loss+error_weight*(misses*mask).sum()/mask.sum().clamp_min(1)
    if presence_weight and output.presence is not None:
        loss = loss+presence_weight*torch.nn.functional.binary_cross_entropy_with_logits(
            output.presence, batch['hand_in_view'].float())
    return loss


# Mixed precision: consecutive overflowing steps GradScaler may skip before training stops.
MAX_SKIPPED_STEPS = 50


def optimizer_step(model, optimizer, scaler, loss, xla=False):
    """Backward, clip to norm 1 and step; returns whether the gradients were finite (on XLA
    as a device flag, read later: reading it now would end the traced step early).

    With mixed precision a scaled fp16 gradient overflows now and then: GradScaler then
    skips the step and lowers its scale, so a non-finite norm is not an error there.
    In full precision it is (a NaN from the model), so it raises; on XLA (float32, no
    scaler) run_epoch raises at its next read instead."""
    optimizer.zero_grad(set_to_none=True)
    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.,
                                          error_if_nonfinite=not scaler.is_enabled() and not xla)
    scaler.step(optimizer)
    scaler.update()
    finite = torch.isfinite(norm)
    return finite if xla else bool(finite)


def _check_finite(nonfinite, phase):
    """XLA: raise when any loss or gradient norm so far was not finite (reads the count)."""
    if int(_number(nonfinite)):
        raise FloatingPointError(f'Nonfinite loss or gradients in {phase} (XLA, float32)')


def run_epoch(model, loader, device, optimizer=None, scaler=None, limit=0, bone_weight=.1, log_every=50,
              relative_weight=0., error_weight=0., presence_weight=0.):
    """One pass over loader; trains when an optimizer is given. Metrics use the final query:
    MPJPE and PCK, the error estimate's miss, and per anchor kind (ANCHOR_KINDS) the share of
    joints with a target, their MPJPE and their anchor's alone (what the correction starts from;
    None for a model without an anchor, HandDirect).

    On XLA (TPU) batches keep their padded shape, float32 throughout, each batch ends with
    sync() and the device is read only every log_every batches and at the end (the first
    batch of each shape also compiles; its time is printed)."""
    training = optimizer is not None
    phase = 'train' if training else 'val'
    xla = is_xla(device)
    model.train(training)
    pose, loss_mean, error_miss = PoseTotals(), WeightedMean(), WeightedMean()
    presence_hit, out_of_view = WeightedMean(), WeightedMean()   # final query, every hand
    kind_rate = {name: WeightedMean() for name in ANCHOR_KINDS}
    kind_mm = {name: WeightedMean() for name in ANCHOR_KINDS}
    kind_anchor_mm = {name: WeightedMean() for name in ANCHOR_KINDS[1:]}
    started, batches, skipped, consecutive, nonfinite = time.monotonic(), 0, 0, 0, 0
    for batch in device_batches(loader, device):
        batch = batch if xla else to_device(trim_padding(batch), device)
        precision = (torch.autocast(device_type='cuda') if training and device.type == 'cuda'
                     else contextlib.nullcontext())
        with torch.set_grad_enabled(training), precision:
            output = model(*model_inputs(batch), return_details=True)
            loss = batch_loss(batch, output, bone_weight, relative_weight, error_weight, presence_weight)
        if xla:
            nonfinite = nonfinite+(~torch.isfinite(loss.detach())).int()
        elif not torch.isfinite(loss):
            raise FloatingPointError('Nonfinite loss')
        if training:
            finite = optimizer_step(model, optimizer, scaler, loss, xla)
            if xla:
                nonfinite = nonfinite+(~finite).int()
            elif finite:
                consecutive = 0
            else:
                skipped, consecutive = skipped+1, consecutive+1
                if consecutive >= MAX_SKIPPED_STEPS:
                    raise FloatingPointError(f'Non-finite gradients in {consecutive} consecutive steps '
                                             f'(grad scale {scaler.get_scale():.3g}): not an fp16 overflow')
        pose.add(output.pose, batch)
        loss_mean.add(loss.detach().float(), len(output.pose))
        final = {key: batch[key][:, -1:] if key != 'world_unit_cm' else batch[key]
                 for key in ('target', 'target_mask', 'world_unit_cm')}
        mm = lambda xyz: torch.expm1(joint_error_log(xyz[:, -1:].detach(), final)[0][:, 0])  # [B,2,21]
        actual = mm(output.pose)
        has_target = batch['target_mask'][:, -1].bool()
        if output.error is not None:
            miss = (torch.expm1(output.error[:, -1].detach().float())-actual).abs()
            error_miss.add((miss*has_target).sum()/has_target.sum().clamp_min(1), has_target.sum())
        in_view = batch['hand_in_view'][:, -1].bool()
        out_of_view.add((~in_view).float().mean(), in_view.numel())
        if output.presence is not None:
            presence_hit.add(((output.presence[:, -1] > 0) == in_view).float().mean(), in_view.numel())
        if output.anchored is not None:                     # HandLiteV3; HandDirect has no anchor
            alone = mm(output.anchor_xyz)
            kind = output.anchored[:, -1]
            for index, name in enumerate(ANCHOR_KINDS):
                chosen = ((kind == index) & has_target).float()
                kind_rate[name].add(chosen.sum()/has_target.sum().clamp_min(1), has_target.sum())
                kind_mm[name].add((actual*chosen).sum()/chosen.sum().clamp_min(1), chosen.sum())
                if index:
                    kind_anchor_mm[name].add((alone*chosen).sum()/chosen.sum().clamp_min(1), chosen.sum())
        sync(device)
        batches += 1
        if xla and batches == 1:
            _check_finite(nonfinite, phase)
            print(f'{phase}: first XLA step (compile + run) {time.monotonic()-started:.1f}s', flush=True)
        if batches % log_every == 0:
            if xla:
                _check_finite(nonfinite, phase)
            print(f"{phase} batch={batches} samples={pose.samples} "
                  f"MPJPE={pose.mpjpe_mm:.2f}mm" + (f' skipped_steps={skipped}' if skipped else ''), flush=True)
        if limit and batches >= limit:
            break
    if xla:
        _check_finite(nonfinite, phase)
    summary = pose.summary()
    return dict(loss=loss_mean.mean, mpjpe_mm=summary['mpjpe_mm'], pck20=summary['pck20'],
                mpjpe_world_mm=summary['mpjpe_world_mm'], mpjpe_rel_mm=summary['mpjpe_rel_mm'],
                error_miss_mm=error_miss.mean, presence_accuracy=presence_hit.mean, out_of_view_rate=out_of_view.mean,
                kind_rate={name: mean.mean for name, mean in kind_rate.items()},
                kind_mpjpe_mm={name: mean.mean for name, mean in kind_mm.items()},
                kind_anchor_mpjpe_mm={name: mean.mean for name, mean in kind_anchor_mm.items()},
                samples=summary['samples'], joints=summary['joints'],
                skipped_steps=skipped, batches=batches, seconds=time.monotonic()-started)


In [ ]:
%%writefile /content/gigahands_training/hand_tracking/events.py
"""Camera events: acceptance, packing and slot selection. No learned parameters."""
import torch
import torch.nn.functional as F
from .constants import NUM_CAMERAS, MODEL_CHANNELS, ARRIVAL_TOLERANCE_S
from .contracts import EventBatch, RawEvents


def accepted_captures(camera, capture, present):
    """Keep strictly newer captures per camera in arrival order; padding is ignored.

    This is the batch equivalent of EventStream's per-camera high-water mark: an event is
    dropped when an earlier present event of its camera has a capture at least as new.
    Pairwise [B,E,E] instead of a running maximum (cummax has no ONNX operator).
    Comparing the original dtype preserves float64 stream timestamp precision.
    """
    present = present.bool() & (camera >= 0) & (camera < NUM_CAMERAS) & torch.isfinite(capture)
    index = torch.arange(camera.shape[1], device=camera.device)
    earlier = index[None, :, None] > index[None, None, :]                                # [1,E(i),E(j)]: j before i
    superseded = (earlier & present[:, None, :] & (camera[:, :, None] == camera[:, None, :])
                  & (capture[:, None, :] >= capture[:, :, None])).any(-1)
    return present & ~superseded


def make_events(features, valid, camera, capture, arrival, present) -> EventBatch:
    present = accepted_captures(camera, capture, present)
    return _pack_events(features, valid, camera, capture, arrival, present)


def _pack_events(features, valid, camera, capture, arrival, present) -> EventBatch:
    """Canonical tensor layout after the caller has resolved event acceptance."""
    valid = valid.bool() & present[..., None, None]
    raw = torch.where(valid[..., None], features[..., :MODEL_CHANNELS].float(), 0.)
    return dict(raw=raw, valid=valid, camera=camera.long(), capture=capture.float(), arrival=arrival.float(),
                present=present.bool())


def relative_events(raw: RawEvents, origin: float) -> EventBatch:
    """Convert one stream's absolute float64 timestamps into a relative model batch."""
    present = torch.ones_like(raw['camera'], dtype=torch.bool)[None]
    # Stream buffers contain only accepted frames; do not rescan history per query.
    return _pack_events(raw['features'][None], raw['valid'][None], raw['camera'][None],
                        (raw['capture']-origin)[None], (raw['arrival']-origin)[None], present)


def _gather(values, index):
    """values [B,E,...], index [B,...] -> values at index along the event axis."""
    batch = torch.arange(values.shape[0], device=values.device).reshape(-1, *[1]*(index.ndim-1))
    return values[batch, index]


def select_slots(events, query, per_camera, span_s):
    """Slots [B,Q,3K]: event indices in arrival order (padding first), and validity. Each
    camera's K latest accepted events that arrived by the query and were captured within
    span_s before it: a fixed-size input whatever the frame rate."""
    b, e = events['camera'].shape
    index = torch.arange(e, device=query.device)
    usable = (events['present'][:, None] & (events['arrival'][:, None] <= query[..., None]+ARRIVAL_TOLERANCE_S)
              & (events['capture'][:, None] >= query[..., None]-span_s))                    # [B,Q,E]
    chosen = []
    for camera in range(NUM_CAMERAS):
        score = torch.where(usable & (events['camera'][:, None] == camera), index, -1)
        if e < per_camera:
            score = F.pad(score, (0, per_camera-e), value=-1)
        chosen.append(score.topk(per_camera, dim=-1).values)
    slots = torch.cat(chosen, -1).sort(-1).values
    return slots.clamp_min(0), slots >= 0


In [ ]:
%%writefile /content/gigahands_training/hand_tracking/geometry.py
"""Differentiable ray geometry: least-squares triangulation and ray-to-point residuals. No learned parameters."""
import torch
from .constants import MIN_RAY_SIN2


def cross3(a, b):
    """Cross product of [...,3] vectors (torch.linalg.cross has no ONNX operator)."""
    a0, a1, a2 = a.unbind(-1)
    b0, b1, b2 = b.unbind(-1)
    return torch.stack((a1*b2-a2*b1, a2*b0-a0*b2, a0*b1-a1*b0), -1)


def solve3(a, b):
    """a [...,3,3] invertible, b [...,3] -> a^-1 b by the adjugate (Cramer's rule): exact for 3x3
    and exportable; callers substitute the identity for singular systems."""
    r0, r1, r2 = a.unbind(-2)
    c0, c1, c2 = cross3(r1, r2), cross3(r2, r0), cross3(r0, r1)
    det = (r0*c0).sum(-1, keepdim=True)
    return (b[..., 0:1]*c0+b[..., 1:2]*c1+b[..., 2:3]*c2)/det


def triangulate(origins, directions, mask):
    """Least-squares point closest to the masked rays: origins/directions [...,C,3], mask [...,C]
    -> point [...,3], valid [...]. Needs two or more rays at least MIN_RAY_SIN2 from parallel and a
    point in front of every used camera; otherwise the point is 0 and valid is False."""
    mask = (mask.bool() & torch.isfinite(origins).all(-1) & torch.isfinite(directions).all(-1)
            & (directions.norm(dim=-1) > 1e-8))
    d = torch.where(mask[..., None], directions, 0.)
    o = torch.where(mask[..., None], origins, 0.)
    d = d/d.norm(dim=-1, keepdim=True).clamp_min(1e-8)
    eye = torch.eye(3, dtype=d.dtype, device=d.device)
    projection = (eye-d[..., :, None]*d[..., None, :])*mask[..., None, None]                 # I - d d^T per ray
    a = projection.sum(-3)
    rhs = (projection@o[..., None]).squeeze(-1).sum(-2)
    # Widest pairwise ray angle instead of eigvalsh (batched eigh can fail on the empty or
    # single-ray matrices present here).
    pair = cross3(d[..., :, None, :], d[..., None, :, :]).square().sum(-1)
    sin2 = (pair*(mask[..., :, None] & mask[..., None, :])).amax((-2, -1))
    valid = (mask.sum(-1) >= 2) & (sin2 > MIN_RAY_SIN2)
    point = solve3(torch.where(valid[..., None, None], a, eye), rhs)
    # A point behind any contributing camera means inconsistent rays, not a hand.
    depth = ((point[..., None, :]-o)*d).sum(-1)
    valid = valid & ((depth > 0) | ~mask).all(-1)
    return torch.where(valid[..., None], point, 0.), valid


def ray_residuals(point, origins, directions):
    """Angular residual (miss distance / depth) of each ray [...,C] to point [...,3]; huge for a
    point behind the camera."""
    d = directions/directions.norm(dim=-1, keepdim=True).clamp_min(1e-8)
    relative = point[..., None, :]-origins
    depth = (relative*d).sum(-1)
    return (relative-depth[..., None]*d).norm(dim=-1)/depth.clamp_min(1e-6)


In [ ]:
%%writefile /content/gigahands_training/hand_tracking/layers.py
"""Network building blocks shared by the models: attention with an additive key bias,
pre-norm blocks and the per-event finger-token encoder. Linear/LayerNorm/GELU/MatMul/
Softmax only and tensors of at most 4 dimensions, so the graphs export to ONNX and ncnn."""
import torch
from torch import nn
import torch.nn.functional as F
from .constants import NUM_CAMERAS, NUM_HANDS, NUM_JOINTS, FINGERS, FINGER_JOINTS, MASK_OFF


def key_bias(valid):
    """Boolean key validity -> additive attention bias (0 or MASK_OFF)."""
    return (valid.float()-1.)*-MASK_OFF


class Attention(nn.Module):
    """Multi-head attention with an additive bias; 4-D tensors at most (ncnn limit)."""

    def __init__(self, dim, heads, dropout):
        super().__init__()
        self.heads, self.dropout = heads, dropout
        self.query, self.key, self.value, self.out = (nn.Linear(dim, dim) for _ in range(4))

    def forward(self, x, keys, bias=None):
        """x [N,L,d], keys [N,S,d], bias broadcastable to [N,heads,L,S] -> [N,L,d]."""
        n, l, d = x.shape
        s, h = keys.shape[1], self.heads
        q = self.query(x).reshape(n, l, h, d//h).permute(0, 2, 1, 3)
        k = self.key(keys).reshape(n, s, h, d//h).permute(0, 2, 3, 1)
        v = self.value(keys).reshape(n, s, h, d//h).permute(0, 2, 1, 3)
        scores = (q @ k)*(d//h)**-.5
        if bias is not None:
            scores = scores+bias
        weights = F.dropout(scores.softmax(-1), self.dropout, self.training)
        return self.out((weights @ v).permute(0, 2, 1, 3).reshape(n, l, d))


class Block(nn.Module):
    """Pre-norm: optional cross-attention to keys, self-attention, feed-forward. dropout on
    the residual branches, attention_dropout on the attention weights."""

    def __init__(self, dim, heads, dropout, cross=True, attention_dropout=0.):
        super().__init__()
        self.cross = Attention(dim, heads, attention_dropout) if cross else None
        if cross:
            self.norm_query, self.norm_keys = nn.LayerNorm(dim), nn.LayerNorm(dim)
        self.norm_self, self.norm_ff = nn.LayerNorm(dim), nn.LayerNorm(dim)
        self.attention = Attention(dim, heads, attention_dropout)
        self.feedforward = nn.Sequential(nn.Linear(dim, 2*dim), nn.GELU(), nn.Linear(2*dim, dim))
        self.drop = nn.Dropout(dropout)

    def forward(self, x, keys=None, bias=None, self_bias=None):
        """bias masks the cross-attention keys, self_bias the self-attention ones."""
        if self.cross is not None:
            x = x+self.drop(self.cross(self.norm_query(x), self.norm_keys(keys), bias))
        y = self.norm_self(x)
        x = x+self.drop(self.attention(y, y, self_bias))
        return x+self.drop(self.feedforward(self.norm_ff(x)))


class EventEncoder(nn.Module):
    """features [N,2,21*F], camera one-hot [N,3] -> tokens [N,2*5,dim] (hand-major, one per
    finger) with finger_tokens, else [N,2,dim] (one per hand).

    Finger tokens share one MLP over the 5 joints (wrist and finger) of a group; a learned
    finger embedding inside the MLP lets each finger be read differently. Groups are taken
    with a constant 0/1 selection matrix (MatMul), which ncnn runs without Gather."""

    def __init__(self, dim, finger_tokens=True, joint_features=14):
        super().__init__()
        self.fingers, self.joint_features = finger_tokens, joint_features
        if finger_tokens:
            group = len(FINGER_JOINTS[0])
            select = torch.zeros(FINGERS*group, NUM_JOINTS)
            select[torch.arange(FINGERS*group), torch.tensor(FINGER_JOINTS).flatten()] = 1.
            self.register_buffer('select', select, persistent=False)
            self.joints = nn.Sequential(nn.Linear(group*joint_features, dim), nn.GELU(), nn.Linear(dim, dim))
            self.finger = nn.Parameter(torch.randn(FINGERS, dim)*.02)
        else:
            self.joints = nn.Sequential(nn.Linear(NUM_JOINTS*joint_features, dim), nn.GELU(), nn.Linear(dim, dim))
        self.hand = nn.Parameter(torch.randn(NUM_HANDS, dim)*.02)
        self.camera = nn.Linear(NUM_CAMERAS, dim, bias=False)
        self.norm = nn.LayerNorm(dim)

    def forward(self, features, camera):
        if not self.fingers:
            return self.norm(self.joints(features)+self.hand+self.camera(camera)[:, None])
        n = features.shape[0]
        groups = self.select@features.reshape(n, NUM_HANDS, NUM_JOINTS, self.joint_features)   # [N,2,5*5,F]
        groups = groups.reshape(n, NUM_HANDS, FINGERS, -1)                                     # [N,2,5,5*F]
        x = self.joints[2](self.joints[1](self.joints[0](groups)+self.finger))
        x = x+self.hand[:, None]+self.camera(camera)[:, None, None]
        return self.norm(x).reshape(n, NUM_HANDS*FINGERS, -1)


In [ ]:
%%writefile /content/gigahands_training/hand_tracking/model.py
"""HandLiteV3: a line-fit triangulation as anchor plus a learned correction, and each joint's
expected error. No camera calibration head and no state carried between queries.

  per event, once, on arrival
    features  u,v, ray origin and direction, capture->arrival delay, detected   [2,21,10]
    encoder   one token per finger of each hand                                [10, dim]
  per query at time t (geometry, no learned parameters, float32)
    rays      per camera and joint, a least-squares line over time through the ray directions
              of the detections within fit_span_s, read at t (one detection, or detections
              too close in time: their mean)
    prior     each joint's own newest triangulation before t: the same line fit and
              triangulation at t - k*prior_step_s (k = 1 .. prior_steps, fit windows that do not
              overlap), the newest that succeeded (observations only: no state, no feedback)
    anchor    triangulated: the least-squares point of the joint's rays. A joint that fails
              now is anchored from its prior point: on the ray (of the cameras that see it)
              passing closest to it, at the ray's point nearest to it (the direction observed
              now, the depth from the past); seeing none, at the prior point; without a prior
              point: zero                                                      [42, 3]
    joints    per joint: anchor, anchor minus prior point, how it was anchored, the prior
              point's age; per camera: detections, fitted, newest/oldest age, newest u,v, the
              ray's miss to the anchor, the fit's shift from the newest ray (an overreaching
              line), the detections' RMS and the newest one's residual from the line (a
              jumping detection)                                                 [42, 56]
  per query (networks)
    slots     each camera's slots_per_camera latest events within event_span_s, their
              tokens told their age (t - capture)                               [10*3K, dim]
    corrector 42 joint tokens (MLP of the joint features + a learned joint embedding):
              [cross-attention to the slot tokens, self-attention over the joints of both
              hands, feed-forward] x blocks -> offset (zero-initialised head)    [42, 3]
    pose      anchor + offset (a joint with no anchor: the offset alone)
    error     (config.error_estimate) per joint, from its final token (detached): its own
              expected error, log(1 + mm), trained on the actual error          [42]

State (LiteV3Stream and the deployment runtime only): each joint's last triangulation however
old, updated whenever it triangulates, and used as the prior point only when the history
search's, so a joint seen by one camera keeps its depth; its age input is capped at
max_prior_age_s. Training (forward()) has no state: its history search reaches prior_span_s back.
Nothing learned is fed back.
"""
import contextlib
import torch
from torch import nn
from .config import LiteV3Config
from .constants import (NUM_CAMERAS, NUM_HANDS, NUM_JOINTS, HAND_JOINTS, FINGERS, FEATURE_CHANNELS, UV, RAY_ORIGIN,
                        RAY_DIRECTION, TIME_UNIT_S, MISS_SCALE, RESIDUAL_UNIT, ARRIVAL_TOLERANCE_S, MIN_TIME_STD_S,
                        MASK_OFF)
from .contracts import ModelOutput
from .events import _gather, make_events, relative_events, select_slots
from .geometry import ray_residuals, triangulate
from .layers import Block, EventEncoder
from .networks import DIRECT_JOINT_FEATURES, DirectEncoded, camera_one_hot, direct_features, repeat_each
from .stream import EventStream

# Per camera: detected, fitted (a slope used), detections / 4, newest and oldest detection age
# (TIME_UNIT_S), newest u,v in [-1,1] (2), the ray's miss vector to the anchor (3) and the fitted
# ray's shift from the newest detection's ray (3), both x MISS_SCALE, the detections' RMS and the
# newest one's angular residual from the line (RESIDUAL_UNIT). Zero for a camera without a detection.
CAMERA_FEATURES = 15
# Per joint: anchor (3), anchor minus its own prior point (3, zero without one), anchored by
# triangulation / on one ray at the prior point's depth / at the prior point (3 flags), a prior
# point exists, the prior point's age (TIME_UNIT_S).
JOINT_FEATURES = 11
LITE3_FEATURES = JOINT_FEATURES+NUM_CAMERAS*CAMERA_FEATURES
# Encoder tokens per event: one per finger of each hand.
EVENT_TOKENS = NUM_HANDS*FINGERS
# ModelOutput.anchored values: no anchor, triangulated, one ray at the prior point's depth, the prior point.
ANCHOR_KINDS = ('none', 'triangulated', 'ray', 'prior')
FAR = 1e6   # a capture time no event has (finite, for XLA)


def float32(device):
    """A region computed in float32 even under CUDA/CPU autocast (XLA trains in float32 anyway)."""
    return torch.autocast(device_type=device.type, enabled=False) if device.type in ('cuda', 'cpu') else contextlib.nullcontext()


def gather_joints(values, pick):
    """values [B,E,42,C], pick [B,...,42] (event index per joint) -> [B,...,42,C]."""
    b, e, joints, c = values.shape
    flat = pick.reshape(b, -1, joints).permute(0, 2, 1)                                     # [B,42,M]
    out = values.permute(0, 2, 1, 3).gather(2, flat[..., None].expand(-1, -1, -1, c))       # [B,42,M,C]
    return out.permute(0, 2, 1, 3).reshape(*pick.shape, c)


def fitted_rays(events, query, span_s, offset_s=0.):
    """Each camera's ray per joint at the reference time query - offset_s, from a least-squares
    line over time through the ray directions of that camera's events in which the joint was
    detected, arrived by the query and captured within span_s before the reference time (not
    after it), read at the reference time (one detection, or detections too close in time:
    their mean). Returns per camera and joint [B,Q,3,42,...]: origin (3, mean), unit direction
    (3), detected, fitted (a slope used), detections, newest and oldest detection age from the
    reference time (s), the newest detection's u,v (2) and unit direction (3), and the
    detections' RMS and the newest one's angular residual from the line (radians)."""
    b, e = events['camera'].shape
    index = torch.arange(e, device=query.device)
    reference = query-offset_s
    usable = (events['present'][:, None] & (events['arrival'][:, None] <= query[..., None]+ARRIVAL_TOLERANCE_S)
              & (events['capture'][:, None] >= reference[..., None]-span_s)
              & (events['capture'][:, None] <= reference[..., None]+ARRIVAL_TOLERANCE_S))     # [B,Q,E]
    detected = events['valid'].reshape(b, 1, e, HAND_JOINTS)
    tau = (events['capture'][:, None]-reference[..., None])[..., None]                      # [B,Q,E,1]
    raw = events['raw'].reshape(b, e, HAND_JOINTS, -1)
    origin, direction = raw[..., RAY_ORIGIN], raw[..., RAY_DIRECTION]                        # [B,E,42,3]
    square = direction.square().sum(-1)                                                     # [B,E,42]
    unit = lambda d: d/d.norm(dim=-1, keepdim=True).clamp_min(1e-8)
    rays = []
    for camera in range(NUM_CAMERAS):
        use = (usable & (events['camera'][:, None] == camera))[..., None] & detected        # [B,Q,E,42]
        m = use.float()
        mt = m*tau
        s0, s1, s2 = m.sum(2), mt.sum(2), (mt*tau).sum(2)                                   # [B,Q,42]
        sd = torch.einsum('bqej,bejc->bqjc', m, direction)
        sdt = torch.einsum('bqej,bejc->bqjc', mt, direction)
        spread = s0*s2-s1*s1                                                                # s0^2 x time variance
        fit = (s0 >= 2) & (spread > MIN_TIME_STD_S**2*s0*s0)
        slope = torch.where(fit[..., None], (s0[..., None]*sdt-s1[..., None]*sd)/spread.clamp_min(1e-12)[..., None], 0.)
        value = (sd-slope*s1[..., None])/s0.clamp_min(1)[..., None]                          # the line at tau = 0
        mean_origin = torch.einsum('bqej,bejc->bqjc', m, origin)/s0.clamp_min(1)[..., None]
        # Sum over detections of |d - value - slope*tau|^2, expanded into the sums above.
        squared = (torch.einsum('bqej,bej->bqj', m, square)-2*(value*sd).sum(-1)-2*(slope*sdt).sum(-1)
                   +value.square().sum(-1)*s0+2*(value*slope).sum(-1)*s1+slope.square().sum(-1)*s2).clamp_min(0.)
        scale = value.norm(dim=-1).clamp_min(1e-8)                                          # residuals as angles
        rms = (squared/s0.clamp_min(1)).sqrt()/scale
        has = use.any(2)
        newest_index = torch.where(use, index[:, None], -1).amax(2).clamp_min(0)            # [B,Q,42]
        newest = gather_joints(raw, newest_index)                                           # [B,Q,42,11]
        newest_tau = torch.where(use, tau, -FAR).amax(2)
        oldest_tau = torch.where(use, tau, FAR).amin(2)
        newest_residual = (newest[..., RAY_DIRECTION]-value-slope*newest_tau.clamp_min(-span_s)[..., None]).norm(dim=-1)/scale
        age = lambda value: torch.where(has, -value, 0.)
        rays.append((mean_origin, unit(value), has, fit, s0, age(newest_tau), age(oldest_tau), newest[..., UV],
                     unit(newest[..., RAY_DIRECTION]), torch.where(has, rms, 0.), torch.where(has, newest_residual, 0.)))
    return tuple(torch.stack(values, 2) for values in zip(*rays))


def rays_at(events, query, config, offset_s=0.):
    """fitted_rays at query - offset_s, per joint and camera [B,Q,2,21,3,...]."""
    b, q = query.shape
    return tuple(value.transpose(2, 3).reshape(b, q, NUM_HANDS, NUM_JOINTS, NUM_CAMERAS, *value.shape[4:])
                 for value in fitted_rays(events, query, config.fit_span_s, offset_s))


def prior_points(events, query, config):
    """Each joint's own newest triangulation before the query: the same line fit and
    triangulation at query - k*prior_step_s (k = 1 .. prior_steps, newest first) -> point
    [B,Q,2,21,3], found [B,Q,2,21] and the age of the newest detection it used (s) [B,Q,2,21]."""
    b, q = query.shape
    point = torch.zeros(b, q, NUM_HANDS, NUM_JOINTS, 3, device=query.device)
    found = torch.zeros(b, q, NUM_HANDS, NUM_JOINTS, dtype=torch.bool, device=query.device)
    age = torch.zeros(b, q, NUM_HANDS, NUM_JOINTS, device=query.device)
    for k in range(1, config.prior_steps+1):
        offset = k*config.prior_step_s
        origin, direction, has, _, _, newest_age = rays_at(events, query, config, offset)[:6]
        p, ok = triangulate(origin, direction, has)
        take = ok & ~found
        point = torch.where(take[..., None], p, point)
        age = torch.where(take, torch.where(has, newest_age, FAR).amin(-1)+offset, age)
        found = found | ok
    return point, found, age


def merge_priors(searched, stored):
    """The history search's prior point per joint, or the stored one where the search found none:
    each (point [...,3], found [...], age (s) [...]). A newer stored point is not preferred, so a
    stream's inputs equal forward()'s (training's) wherever the search finds a point."""
    (point, found, age), (kept, known, kept_age) = searched, stored
    use = known & ~found
    return (torch.where(use[..., None], kept, point), found | known, torch.where(use, kept_age, age))


def anchor_features(events, query, config, stored=None):
    """Anchor [B,Q,2,21,3], how each joint was anchored (ANCHOR_KINDS index) [B,Q,2,21], the
    corrector's joint input [B,Q,42,F] and the age of the newest detection each triangulated
    joint used (s) [B,Q,2,21] (a stream's state update).

    The prior point is the newest triangulation found by prior_points in the window's history,
    or, where it finds none, stored (point, found, age): a stream's state, the joint's last
    triangulation however old. Its age input is capped at max_prior_age_s (the oldest training has shown)."""
    b, q = query.shape
    origin, direction, has, fit, count, age, oldest, uv, newest_direction, rms, residual = rays_at(events, query, config)
    point, ok = triangulate(origin, direction, has)                                         # [B,Q,2,21,3], [B,Q,2,21]
    prior, found, prior_age = prior_points(events, query, config)                       # the joint's own past
    if stored is not None:
        prior, found, prior_age = merge_priors((prior, found, prior_age), stored)
    observed = torch.where(has, age, FAR).amin(-1)                                          # newest detection used now
    # A joint that fails now: the point of the ray passing closest to its prior point, nearest to it.
    miss = torch.where(has, ray_residuals(prior, origin, direction), FAR)                # [B,Q,2,21,3]
    best = miss.argmin(-1, keepdim=True)
    pick = lambda value: value.gather(-2, best[..., None].expand(*best.shape, 3))[..., 0, :]
    o, d = pick(origin), pick(direction)
    projected = o+((prior-o)*d).sum(-1, keepdim=True).clamp_min(0.)*d
    seen = has.any(-1)
    on_ray = ~ok & found & seen
    at_prior = ~ok & found & ~seen
    anchor = torch.where(ok[..., None], point, torch.where(on_ray[..., None], projected,
                                                           torch.where(at_prior[..., None], prior, 0.)))
    kind = ok.long()+2*on_ray.long()+3*at_prior.long()
    relative = anchor[..., None, :]-origin
    miss_vector = relative-direction*(relative*direction).sum(-1, keepdim=True)
    flag = lambda value: value[..., None].float()
    camera = torch.cat((flag(has), flag(fit), count[..., None]/4, age[..., None]/TIME_UNIT_S, oldest[..., None]/TIME_UNIT_S,
                        uv*2-1, miss_vector*MISS_SCALE, (direction-newest_direction)*MISS_SCALE,
                        rms[..., None]/RESIDUAL_UNIT, residual[..., None]/RESIDUAL_UNIT), -1)
    camera = torch.where(has[..., None], camera, 0.)                                        # [B,Q,2,21,3,15]
    prior_age = torch.where(found, prior_age.clamp_max(config.max_prior_age_s), 0.)
    joint = torch.cat((anchor, torch.where(found[..., None], anchor-prior, 0.), flag(ok), flag(on_ray),
                       flag(at_prior), flag(found), prior_age[..., None]/TIME_UNIT_S), -1)  # [B,Q,2,21,11]
    features = torch.cat((joint, camera.reshape(*camera.shape[:-2], -1)), -1)
    return anchor, kind, features.reshape(b, q, HAND_JOINTS, -1), torch.where(ok, observed, 0.)


def head(dim, out, zero=False):
    """Joint tokens [...,dim] -> [...,out]; zero: starts at zero output."""
    layers = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, dim), nn.GELU(), nn.Linear(dim, out))
    if zero:
        nn.init.zeros_(layers[-1].weight)
        nn.init.zeros_(layers[-1].bias)
    return layers


class Corrector(nn.Module):
    """Joint features [N,42,F], slot tokens [N,T,dim], validity [N,T] (1/0) and ages [N,T]
    (TIME_UNIT_S) -> offsets [N,42,3], each joint's expected error log(1 + mm) [N,42] (None
    without error) and each hand's logit of being inside some camera's view [N,2] (None without
    presence).

    Per-joint MLP plus a learned joint embedding; each block cross-attends to the slot tokens
    (with an age embedding, and a null key for a query without slots), then self-attends over
    the 42 joints of both hands; zero-initialised offset head; the error head (per joint) and the
    presence head (per hand, on the mean of its joints' tokens) read the final tokens detached,
    so learning them never changes the pose."""

    def __init__(self, features, dim, heads, blocks, dropout, attention_dropout, error=True, presence=True):
        super().__init__()
        self.input = nn.Sequential(nn.Linear(features, dim), nn.GELU(), nn.Linear(dim, dim))
        self.joints = nn.Parameter(torch.randn(HAND_JOINTS, dim)*.02)
        self.age = nn.Sequential(nn.Linear(1, 16), nn.GELU(), nn.Linear(16, dim))
        self.null = nn.Parameter(torch.zeros(1, dim))
        self.blocks = nn.ModuleList([Block(dim, heads, dropout, attention_dropout=attention_dropout)
                                     for _ in range(blocks)])
        self.output = head(dim, 3, zero=True)
        self.error = head(dim, 1) if error else None
        self.presence = head(dim, 1) if presence else None

    def forward(self, features, tokens, valid, ages):
        x = self.input(features)+self.joints
        n, _, d = tokens.shape
        keys = torch.cat((self.null.expand(n, 1, d), (tokens+self.age(ages[..., None]))*valid[..., None]), 1)
        bias = torch.cat((torch.zeros_like(valid[:, :1]), (valid-1.)*-MASK_OFF), -1)[:, None, None]
        for block in self.blocks:
            x = block(x, keys, bias)
        error = None if self.error is None else self.error(x.detach())[..., 0]
        presence = None if self.presence is None else self.presence(
            x.detach().reshape(n, NUM_HANDS, NUM_JOINTS, -1).mean(2))[..., 0]
        return self.output(x), error, presence


class CorrectorGraph(nn.Module):
    """The exported corrector: offsets, then (with the error head) each joint's expected error and
    (with the presence head) its hand's in-view logit repeated on the hand's joints, as columns
    [N,42,3 to 5] (the runtime reads one output per graph)."""

    def __init__(self, corrector):
        super().__init__()
        self.corrector = corrector

    def forward(self, features, tokens, valid, ages):
        offset, error, presence = self.corrector(features, tokens, valid, ages)
        columns = [offset]
        if error is not None:
            columns.append(error[..., None])
        if presence is not None:
            n = presence.shape[0]
            columns.append(presence[..., None].expand(n, NUM_HANDS, NUM_JOINTS).reshape(n, HAND_JOINTS, 1))
        return torch.cat(columns, -1) if len(columns) > 1 else offset


class HandLiteV3(nn.Module):
    def __init__(self, config=None):
        super().__init__()
        self.config = config or LiteV3Config()
        c = self.config
        self.encoder = EventEncoder(c.dim, True, DIRECT_JOINT_FEATURES)
        self.corrector = Corrector(LITE3_FEATURES, c.dim, c.heads, c.blocks, c.dropout, c.attention_dropout,
                                   c.error_estimate, c.presence)

    @property
    def slots(self):
        return NUM_CAMERAS*self.config.slots_per_camera

    def encode_events(self, events, targets=None):
        """Per-event finger tokens (DirectEncoded [B,T,10,dim]) of the target events (default:
        all), each from its own frame only."""
        b, e = events['camera'].shape
        if targets is None:
            targets = torch.arange(e, device=events['camera'].device)
        t = targets.numel()
        features = direct_features(events['raw'][:, targets], events['valid'][:, targets])
        camera = camera_one_hot(events['camera'][:, targets])
        tokens = self.encoder(features.reshape(b*t, NUM_HANDS, -1), camera.reshape(b*t, NUM_CAMERAS))
        return DirectEncoded(tokens.reshape(b, t, *tokens.shape[1:]))

    def slot_tokens(self, events, encoded, query):
        """Slot tokens [N,10*3K,dim], validity [N,10*3K] (1/0), ages [N,10*3K] (TIME_UNIT_S)
        and whether each query had any slot [B,Q]."""
        b, q = query.shape
        n, s = b*q, self.slots
        slot, valid = select_slots(events, query, self.config.slots_per_camera, self.config.event_span_s)
        take = lambda value: _gather(value, slot).reshape(n, s, *value.shape[2:])
        valid = valid.reshape(n, s)
        token_valid = repeat_each(valid, EVENT_TOKENS)
        tokens = torch.where(token_valid[..., None], take(encoded.tokens).reshape(n, EVENT_TOKENS*s, -1), 0.)
        ages = torch.where(valid, ((query.reshape(n, 1)-take(events['capture']))/TIME_UNIT_S).clamp_min(0.), 0.)
        return tokens, token_valid.float(), repeat_each(ages, EVENT_TOKENS), valid.any(-1).reshape(b, q)

    def estimate(self, events, encoded, query, stored=None):
        """Pose [B,Q,2,21,3], whether a query had any usable frame or slot [B,Q], each joint's
        expected error log(1 + mm) [B,Q,2,21] (None without config.error_estimate), each hand's
        in-view logit [B,Q,2] (None without config.presence), how each joint was anchored
        (ANCHOR_KINDS index) [B,Q,2,21], its anchor [B,Q,2,21,3] and, where triangulated,
        the age of the newest detection used (s) [B,Q,2,21]. stored: a stream's state (see
        anchor_features). The geometry runs in float32 even under autocast."""
        b, q = query.shape
        c = self.config
        with torch.no_grad(), float32(query.device):                                        # no learned parameters
            anchor, kind, features, observed = anchor_features(events, query.float(), c, stored)
        usable = (events['present'][:, None] & (events['arrival'][:, None] <= query[..., None]+ARRIVAL_TOLERANCE_S)
                  & (events['capture'][:, None] >= query[..., None]-c.fit_span_s)).any(-1)
        tokens, token_valid, ages, any_slot = self.slot_tokens(events, encoded, query)
        offset, error, presence = self.corrector(features.reshape(b*q, HAND_JOINTS, -1), tokens, token_valid, ages)
        pose = anchor+offset.float().reshape(b, q, NUM_HANDS, NUM_JOINTS, 3)
        if error is not None:
            error = error.float().reshape(b, q, NUM_HANDS, NUM_JOINTS)
        if presence is not None:
            presence = presence.float().reshape(b, q, NUM_HANDS)
        return pose, usable | any_slot, error, presence, kind, anchor, observed

    def decode_queries(self, events, encoded, query):
        """Without a state: pose [B,Q,2,21,3], None (no camera calibration), accepted [B,Q], None
        (one stage), expected error, anchor kind, anchor and in-view logit (see estimate)."""
        pose, accepted, error, presence, kind, anchor, _ = self.estimate(events, encoded, query)
        return pose, None, accepted, None, error, kind, anchor, presence

    def forward(self, event_features, event_valid, event_camera, event_capture, event_arrival, event_present,
                query_times, return_details=False):
        """Model inputs (contracts.MODEL_INPUT_KEYS). return_details gives ModelOutput."""
        if event_features.ndim != 5 or event_features.shape[2:] != (NUM_HANDS, NUM_JOINTS, FEATURE_CHANNELS):
            raise ValueError('Expected event features [B,E,2,21,14]')
        if event_valid.shape != event_features.shape[:-1] or query_times.ndim != 2:
            raise ValueError('Invalid event/query dimensions')
        events = make_events(event_features, event_valid, event_camera, event_capture, event_arrival, event_present)
        output = ModelOutput(*self.decode_queries(events, self.encode_events(events), query_times.float()))
        return output if return_details else output.pose


class LiteV3Stream(EventStream):
    """HandLiteV3 event by event: each event's tokens are computed once on arrival; a query runs
    the geometry and the corrector.

    State: each joint's last triangulation (point, and the absolute capture time of the newest
    detection it used), updated whenever the joint triangulates. A query uses it as the prior
    point only when the history search finds none, so a joint seen by one camera keeps its depth
    however long; wherever the search finds one, the inputs (and pose) equal forward()'s, as in
    training. Nothing learned is fed back."""

    def reset(self):
        super().reset()
        shape = (NUM_HANDS, NUM_JOINTS)
        self.prior_point = torch.zeros(*shape, 3, device=self.device)
        self.prior_capture = torch.zeros(shape, dtype=torch.float64, device=self.device)
        self.prior_known = torch.zeros(shape, dtype=torch.bool, device=self.device)

    def decode(self, time):
        encoded = self.store.encoded.map(lambda value: value[None])
        query = torch.zeros(1, 1, device=self.device)
        stored = (self.prior_point[None, None], self.prior_known[None, None],
                  (time-self.prior_capture).float()[None, None])
        pose, _, _, _, kind, anchor, observed = self.model.estimate(relative_events(self.store.raw, time), encoded, query, stored)
        triangulated = kind[0, 0] == 1
        self.prior_point = torch.where(triangulated[..., None], anchor[0, 0], self.prior_point)
        self.prior_capture = torch.where(triangulated, time-observed[0, 0].double(), self.prior_capture)
        self.prior_known = self.prior_known | triangulated
        return pose[0, 0]


In [ ]:
%%writefile /content/gigahands_training/hand_tracking/networks.py
"""Helpers shared by the model and its export: per-event encoder input, camera one-hot, fixed-count
repeats and the per-event token store."""
from dataclasses import dataclass
import torch
from .constants import NUM_CAMERAS, UV, RAY_ORIGIN, RAY_DIRECTION, DELAY, TIME_UNIT_S

# Per joint: u,v (2), ray origin (3) and direction (3), delay (1), detected (1).
DIRECT_JOINT_FEATURES = 10


def direct_features(raw, valid):
    """Event features [...,2,21,11] (invalid joints zeroed), valid -> encoder input [...,2,21,10]."""
    x = torch.cat((raw[..., UV]*2-1, raw[..., RAY_ORIGIN], raw[..., RAY_DIRECTION], raw[..., DELAY]/TIME_UNIT_S,
                   valid.float()[..., None]), -1)
    return torch.where(valid[..., None], x, 0.)


def camera_one_hot(camera):
    """Camera index [...] -> one-hot [...,3] by comparison (F.one_hot checks the index range
    on the host, a device wait under XLA)."""
    return (camera[..., None] == torch.arange(NUM_CAMERAS, device=camera.device)).float()


def repeat_each(value, count):
    """[...,S] -> [...,S*count], each element repeated count times in place (repeat_interleave
    with a fixed count, as expand + reshape: a static shape under XLA)."""
    return value[..., None].expand(*value.shape, count).reshape(*value.shape[:-1], -1)


@dataclass
class DirectEncoded:
    """Per-event tokens [B,E,...,dim]; they never change after arrival."""
    tokens: torch.Tensor

    def map(self, fn):
        return DirectEncoded(fn(self.tokens))

    def append(self, other, dim=1):
        return DirectEncoded(torch.cat((self.tokens, other.tokens), dim))


In [ ]:
%%writefile /content/gigahands_training/hand_tracking/objectives.py
"""Training losses (pose, wrist-relative pose, bone length, calibration auxiliary) and evaluation totals."""
from dataclasses import dataclass
import math
import torch
import torch.nn.functional as F


def calibration_loss(prediction, target, rotation_deg=5., position_std=.1):
    """Mean squared calibration error in prior-std units over cameras with a known target.

    prediction/target [N,3,6]; target rows of NaN (e.g. real captures, padding) are ignored.
    The common rig motion is unobservable, so the optimum is its posterior mean.
    """
    scale = prediction.new_tensor([math.radians(rotation_deg)]*3+[position_std]*3)
    known = torch.isfinite(target).all(-1)
    error = ((prediction-torch.where(known[...,None], target, 0.))/scale).square().mean(-1)
    return (error*known).sum()/known.sum().clamp_min(1)


BONES = [(0,1+4*f) for f in range(5)] + [(1+4*f+j,2+4*f+j) for f in range(5) for j in range(3)]


def pose_loss(prediction, target, target_mask, world_unit_cm, bone_weight=.1, beta_m=.006):
    """Masked SmoothL1 plus target-specific bone length loss, in meters.

    prediction/target [B,...,2,21,3]; world_unit_cm [B] converts each sample, so
    beta and the loss scale do not depend on the export's world unit.
    """
    valid = target_mask.bool()
    scale = (world_unit_cm/100).reshape(-1, *[1]*(prediction.ndim-1))
    prediction = prediction.float()*scale
    safe_target = torch.where(valid[...,None], target, 0.)*scale
    joints = F.smooth_l1_loss(prediction, safe_target, beta=beta_m, reduction='none').mean(-1)
    position = (joints*valid).sum()/valid.sum().clamp_min(1)
    a, b = map(list, zip(*BONES))
    pred_lengths = (prediction[...,a,:]-prediction[...,b,:]).norm(dim=-1)
    true_lengths = (safe_target[...,a,:]-safe_target[...,b,:]).norm(dim=-1)
    bone_mask = valid[...,a] & valid[...,b]
    bone = ((pred_lengths-true_lengths).abs()*bone_mask).sum()/bone_mask.sum().clamp_min(1)
    return position+bone_weight*bone


def wrist_relative(joints, mask):
    """Joints [...,21,3] and mask [...,21] -> each non-wrist joint minus its hand's wrist
    [...,20,3] and where both have a target [...,20]."""
    return joints[...,1:,:]-joints[...,:1,:], mask[...,1:] & mask[...,:1]


def relative_loss(prediction, target, target_mask, world_unit_cm, beta_m=.006):
    """Masked SmoothL1 of the joints relative to their wrist (the hand's shape), in meters.

    The wrist position error cancels, so hand shape keeps its own training signal; used
    beside pose_loss, which keeps the absolute position (and hands without a wrist target).
    """
    valid = target_mask.bool()
    scale = (world_unit_cm/100).reshape(-1, *[1]*(prediction.ndim-1))
    prediction, mask = wrist_relative(prediction.float()*scale, valid)
    safe_target, _ = wrist_relative(torch.where(valid[...,None], target, 0.)*scale, valid)
    joints = F.smooth_l1_loss(prediction, safe_target, beta=beta_m, reduction='none').mean(-1)
    return (joints*mask).sum()/mask.sum().clamp_min(1)


def error_totals(prediction, target, mask, world_unit_cm):
    """MPJPE numerator (mm), PCK@20mm numerator and valid joint count for [B,2,21,3]."""
    valid = mask.bool()
    distance = (prediction-torch.where(valid[...,None],target,0.)).norm(dim=-1)
    mm = distance*world_unit_cm[:,None,None]*10
    return (mm*valid).sum(), ((mm<20)&valid).sum(), valid.sum()


def relative_error_totals(prediction, target, mask, world_unit_cm):
    """Wrist-relative MPJPE numerator (mm) and joint count for [B,2,21,3] (hand shape only)."""
    valid = mask.bool()
    prediction, relative_mask = wrist_relative(prediction, valid)
    target, _ = wrist_relative(torch.where(valid[...,None], target, 0.), valid)
    mm = (prediction-target).norm(dim=-1)*world_unit_cm[:,None,None]*10
    return (mm*relative_mask).sum(), relative_mask.sum()


def _number(value):
    """A running total as a Python number; a device tensor is read (waited for) only here."""
    return value.item() if isinstance(value, torch.Tensor) else value


@dataclass
class PoseTotals:
    """Running MPJPE and PCK@20mm over the final query of each window (the reported metric),
    against the target frame's targets, MPJPE against the world-frame targets, and the
    wrist-relative MPJPE (hand shape, wrist position error removed).

    Totals stay device tensors between reads, so adding a batch never waits for the device
    (under XLA, a read ends the traced step); mpjpe_mm and summary() read them."""
    error_mm: float = 0.
    correct: int = 0
    joints: int = 0
    samples: int = 0
    world_error_mm: float = 0.
    relative_error_mm: float = 0.
    relative_joints: int = 0

    def add(self, pose, batch):
        """pose [B,Q,2,21,3] against a batch's target/target_mask/world_unit_cm."""
        with torch.no_grad():
            error, correct, joints = error_totals(pose[:,-1].float(), batch['target'][:,-1],
                                                  batch['target_mask'][:,-1], batch['world_unit_cm'])
            world, _, _ = error_totals(pose[:,-1].float(), batch['target_world'][:,-1], batch['target_mask'][:,-1],
                                       batch['world_unit_cm'])
            relative, relative_joints = relative_error_totals(pose[:,-1].float(), batch['target'][:,-1],
                                                              batch['target_mask'][:,-1], batch['world_unit_cm'])
        self.error_mm = self.error_mm+error
        self.world_error_mm = self.world_error_mm+world
        self.relative_error_mm = self.relative_error_mm+relative
        self.relative_joints = self.relative_joints+relative_joints
        self.correct = self.correct+correct
        self.joints = self.joints+joints
        self.samples += len(pose)

    @property
    def mpjpe_mm(self):
        joints = _number(self.joints)
        return _number(self.error_mm)/joints if joints else float('nan')

    def summary(self):
        joints, relative_joints = int(_number(self.joints)), int(_number(self.relative_joints))
        if not joints:
            raise ValueError('No valid observations/targets in this loader')
        relative = _number(self.relative_error_mm)/relative_joints if relative_joints else float('nan')
        return dict(mpjpe_mm=_number(self.error_mm)/joints, pck20=_number(self.correct)/joints,
                    mpjpe_world_mm=_number(self.world_error_mm)/joints, mpjpe_rel_mm=relative,
                    samples=self.samples, joints=joints)


@dataclass
class WeightedMean:
    """Weighted running mean; value/weight may be device tensors, read only by mean."""
    total: float = 0.
    weight: float = 0.

    def add(self, value, weight):
        self.total = self.total+value*weight
        self.weight = self.weight+weight

    @property
    def mean(self):
        """None when nothing was weighted (e.g. no known calibration target)."""
        weight = _number(self.weight)
        return _number(self.total)/weight if weight else None


In [ ]:
%%writefile /content/gigahands_training/hand_tracking/runtime.py
"""HandLiteV3 deployment runtime without PyTorch: the geometry (line-fit rays, triangulation,
anchors, joint features) and slot selection in numpy around the exported ncnn networks (encoder
per event, corrector per query). Mirrors hand_tracking.model.LiteV3Stream; no state besides the
recent events and each joint's last triangulation (the state: a joint seen by one camera keeps its
depth however long).

    runtime = LiteV3Runtime('deploy_dir')
    runtime.push(camera, features [2,21,14], valid [2,21], capture_time, arrival_time)
    pose = runtime.query(time)          # [2,21,3] world units
    runtime.error_mm                    # [2,21] each joint's expected error in mm (or None)
    runtime.anchor_kind                 # [2,21] how each joint was anchored (model.ANCHOR_KINDS index)
    runtime.in_view_probability         # [2] each hand's probability of being inside some camera's view
                                        #     (below 0.5: out of all cameras, ignore that hand's joints)

Times are absolute seconds (float64). Needs numpy and `ncnn`.
"""
from dataclasses import dataclass
import json
from pathlib import Path
import numpy as np
from .config import saved_config
from .constants import (NUM_CAMERAS, NUM_HANDS, NUM_JOINTS, HAND_JOINTS, FINGERS, MODEL_CHANNELS, UV, RAY_ORIGIN,
                        RAY_DIRECTION, DELAY, TIME_UNIT_S, MISS_SCALE, RESIDUAL_UNIT, STREAM_KEEP_MARGIN_S,
                        ARRIVAL_TOLERANCE_S, MIN_TIME_STD_S, MIN_RAY_SIN2, check_event)

META_FILE = 'litev3.json'
# 2: failed joints anchored from their own last triangulation (prior_span_s, prior_step_s); 3: plus the state
# (each joint's last triangulation however old), prior age capped at max_prior_age_s; 4: each hand's in-view
# logit as the corrector's last column (config.presence)
EXPORT_FORMAT = 4
GRAPHS = ('encoder', 'corrector')
EVENT_TOKENS = NUM_HANDS*FINGERS     # finger tokens per event
FAR = 1e6


def direct_features(raw, valid):
    """Event features [...,2,21,11] (invalid joints zeroed), valid -> encoder input [...,2,21,10]
    (networks.direct_features)."""
    x = np.concatenate((raw[..., UV]*2-1, raw[..., RAY_ORIGIN], raw[..., RAY_DIRECTION], raw[..., DELAY]/TIME_UNIT_S,
                        valid[..., None]), -1)
    return np.where(valid[..., None], x, 0.)


def triangulate(origins, directions, mask):
    """origins/directions [...,C,3], mask [...,C] -> point [...,3], valid [...] (geometry.triangulate)."""
    mask = (mask & np.isfinite(origins).all(-1) & np.isfinite(directions).all(-1)
            & (np.linalg.norm(directions, axis=-1) > 1e-8))
    d = np.where(mask[..., None], directions, 0.)
    o = np.where(mask[..., None], origins, 0.)
    d = d/np.maximum(np.linalg.norm(d, axis=-1, keepdims=True), 1e-8)
    count = mask.sum(-1)
    a = count[..., None, None]*np.eye(3)-np.einsum('...ci,...cj->...ij', d, d)
    rhs = (o-d*(d*o).sum(-1, keepdims=True)).sum(-2)
    sin2 = np.zeros(count.shape)
    for i in range(d.shape[-2]):
        for j in range(i+1, d.shape[-2]):
            sin2 = np.maximum(sin2, np.square(np.cross(d[..., i, :], d[..., j, :])).sum(-1))
    valid = (count >= 2) & (sin2 > MIN_RAY_SIN2)
    point = np.linalg.solve(np.where(valid[..., None, None], a, np.eye(3)), rhs[..., None])[..., 0]
    depth = ((point[..., None, :]-o)*d).sum(-1)
    valid = valid & ((depth > 0) | ~mask).all(-1)
    return np.where(valid[..., None], point, 0.), valid


def ray_residuals(point, origins, directions):
    """Angular residual of each ray [...,C] to point [...,3] (geometry.ray_residuals)."""
    d = directions/np.maximum(np.linalg.norm(directions, axis=-1, keepdims=True), 1e-8)
    relative = point[..., None, :]-origins
    depth = (relative*d).sum(-1)
    return np.linalg.norm(relative-depth[..., None]*d, axis=-1)/np.maximum(depth, 1e-6)


def unit(d):
    return d/np.maximum(np.linalg.norm(d, axis=-1, keepdims=True), 1e-8)


def fitted_rays(events, time, span_s, offset_s=0.):
    """model.fitted_rays for one query at time - offset_s: per camera and joint [3,42,...]."""
    c, j = NUM_CAMERAS, HAND_JOINTS
    reference = time-offset_s
    origin, direction, newest_direction = np.zeros((c, j, 3)), np.zeros((c, j, 3)), np.zeros((c, j, 3))
    has, fit = np.zeros((c, j), bool), np.zeros((c, j), bool)
    count, age, oldest, rms, residual = (np.zeros((c, j)) for _ in range(5))
    uv = np.zeros((c, j, 2))
    for camera in range(c):
        usable = [e for e in events if e.camera == camera and e.arrival <= time+ARRIVAL_TOLERANCE_S
                  and reference-span_s <= e.capture <= reference+ARRIVAL_TOLERANCE_S]
        if not usable:
            continue
        use = np.stack([e.valid.reshape(j) for e in usable])                               # [n,42]
        m = use.astype(np.float64)
        tau = np.array([e.capture-reference for e in usable])[:, None]                    # [n,1]
        raw = np.stack([e.raw.reshape(j, -1) for e in usable])                             # [n,42,11]
        d, o = raw[..., RAY_DIRECTION], raw[..., RAY_ORIGIN]
        mt = m*tau
        s0, s1, s2 = m.sum(0), mt.sum(0), (mt*tau).sum(0)
        sd, sdt = (m[..., None]*d).sum(0), (mt[..., None]*d).sum(0)
        spread = s0*s2-s1*s1
        fitted = (s0 >= 2) & (spread > MIN_TIME_STD_S**2*s0*s0)
        slope = np.where(fitted[:, None], (s0[:, None]*sdt-s1[:, None]*sd)/np.maximum(spread, 1e-12)[:, None], 0.)
        value = (sd-slope*s1[:, None])/np.maximum(s0, 1)[:, None]                         # the line at the query time
        squared = np.maximum((m*np.square(d).sum(-1)).sum(0)-2*(value*sd).sum(-1)-2*(slope*sdt).sum(-1)
                             +np.square(value).sum(-1)*s0+2*(value*slope).sum(-1)*s1+np.square(slope).sum(-1)*s2, 0.)
        scale = np.maximum(np.linalg.norm(value, axis=-1), 1e-8)
        seen = use.any(0)
        newest_index = np.where(use, np.arange(len(usable))[:, None], -1).max(0).clip(0)   # [42]
        newest = raw[newest_index, np.arange(j)]                                          # [42,11]
        newest_tau, oldest_tau = np.where(use, tau, -FAR).max(0), np.where(use, tau, FAR).min(0)
        newest_residual = np.linalg.norm(newest[:, RAY_DIRECTION]-value-slope*np.maximum(newest_tau, -span_s)[:, None],
                                         axis=-1)/scale
        origin[camera] = (m[..., None]*o).sum(0)/np.maximum(s0, 1)[:, None]
        direction[camera], has[camera], fit[camera], count[camera] = unit(value), seen, fitted, s0
        age[camera], oldest[camera] = np.where(seen, -newest_tau, 0.), np.where(seen, -oldest_tau, 0.)
        uv[camera], newest_direction[camera] = newest[:, UV], unit(newest[:, RAY_DIRECTION])
        rms[camera] = np.where(seen, np.sqrt(squared/np.maximum(s0, 1))/scale, 0.)
        residual[camera] = np.where(seen, newest_residual, 0.)
    return origin, direction, has, fit, count, age, oldest, uv, newest_direction, rms, residual


def rays_at(events, time, config, offset_s=0.):
    """model.rays_at for one query: fitted_rays per joint and camera [2,21,3,...]."""
    return tuple(value.swapaxes(0, 1).reshape(NUM_HANDS, NUM_JOINTS, NUM_CAMERAS, *value.shape[2:])
                 for value in fitted_rays(events, time, config.fit_span_s, offset_s))


def prior_points(events, time, config):
    """model.prior_points for one query -> point [2,21,3], found [2,21], age (s) [2,21]."""
    point, found, age = np.zeros((NUM_HANDS, NUM_JOINTS, 3)), np.zeros((NUM_HANDS, NUM_JOINTS), bool), \
        np.zeros((NUM_HANDS, NUM_JOINTS))
    for k in range(1, config.prior_steps+1):
        offset = k*config.prior_step_s
        origin, direction, has, _, _, newest_age = rays_at(events, time, config, offset)[:6]
        p, ok = triangulate(origin, direction, has)
        take = ok & ~found
        point = np.where(take[..., None], p, point)
        age = np.where(take, np.where(has, newest_age, FAR).min(-1)+offset, age)
        found = found | ok
    return point, found, age


def anchor_features(events, time, config, stored=None):
    """model.anchor_features for one query -> anchor [2,21,3], kind [2,21], joint features [42,F]
    and where triangulated the age of the newest detection used (s) [2,21]. stored: the state's
    (point [2,21,3], known [2,21], age (s) [2,21]), used only where the history search finds none."""
    origin, direction, has, fit, count, age, oldest, uv, newest_direction, rms, residual = rays_at(events, time, config)
    point, ok = triangulate(origin, direction, has)
    prior, found, prior_age = prior_points(events, time, config)
    if stored is not None:
        kept, known, kept_age = stored
        use = known & ~found
        prior, found, prior_age = np.where(use[..., None], kept, prior), found | known, np.where(use, kept_age, prior_age)
    observed = np.where(has, age, FAR).min(-1)
    prior_age = np.where(found, np.minimum(prior_age, config.max_prior_age_s), 0.)
    miss = np.where(has, ray_residuals(prior, origin, direction), FAR)                      # [2,21,3]
    best = miss.argmin(-1)
    o = np.take_along_axis(origin, best[..., None, None], -2)[..., 0, :]
    dd = np.take_along_axis(direction, best[..., None, None], -2)[..., 0, :]
    projected = o+np.maximum(((prior-o)*dd).sum(-1, keepdims=True), 0.)*dd
    seen = has.any(-1)
    on_ray, at_prior = ~ok & found & seen, ~ok & found & ~seen
    anchor = np.where(ok[..., None], point, np.where(on_ray[..., None], projected,
                                                     np.where(at_prior[..., None], prior, 0.)))
    kind = ok*1+on_ray*2+at_prior*3
    relative = anchor[..., None, :]-origin
    miss_vector = relative-direction*(relative*direction).sum(-1, keepdims=True)
    camera = np.concatenate((has[..., None], fit[..., None], count[..., None]/4, age[..., None]/TIME_UNIT_S,
                             oldest[..., None]/TIME_UNIT_S, uv*2-1, miss_vector*MISS_SCALE,
                             (direction-newest_direction)*MISS_SCALE, rms[..., None]/RESIDUAL_UNIT,
                             residual[..., None]/RESIDUAL_UNIT), -1)
    camera = np.where(has[..., None], camera, 0.)                                           # [2,21,3,15]
    joint = np.concatenate((anchor, np.where(found[..., None], anchor-prior, 0.), ok[..., None], on_ray[..., None],
                            at_prior[..., None], found[..., None], prior_age[..., None]/TIME_UNIT_S), -1)
    features = np.concatenate((joint, camera.reshape(NUM_HANDS, NUM_JOINTS, -1)), -1).astype(np.float64)
    return anchor, kind, features.reshape(HAND_JOINTS, -1), np.where(ok, observed, 0.)


def select_slot_events(events, time, per_camera, span_s):
    """Each camera's per_camera latest events (with .camera/.capture/.arrival, in arrival
    order) arrived by time and captured within span_s, in arrival order; None for padding
    (first). events.select_slots for one query."""
    chosen = [[] for _ in range(NUM_CAMERAS)]
    for index in range(len(events)-1, -1, -1):
        event = events[index]
        if event.arrival <= time+ARRIVAL_TOLERANCE_S and event.capture >= time-span_s \
                and len(chosen[event.camera]) < per_camera:
            chosen[event.camera].append(index)
    ordered = sorted(i for indices in chosen for i in indices)
    return [None]*(NUM_CAMERAS*per_camera-len(ordered))+[events[i] for i in ordered]


class NcnnGraphs:
    """ncnn nets. Inputs are copied into ncnn-owned Mats and kept alive until extract():
    the Python binding crashes when an in-place layer writes into numpy-backed memory."""

    def __init__(self, directory, names, threads=1, fp16=True):
        import ncnn
        self.ncnn = ncnn
        self.nets = {}
        for name in names:
            net = ncnn.Net()
            net.opt.use_vulkan_compute = False
            for option in ('use_fp16_packed', 'use_fp16_storage', 'use_fp16_arithmetic'):
                setattr(net.opt, option, fp16)
            if threads:
                net.opt.num_threads = threads
            prefix = str(Path(directory)/name)
            if net.load_param(prefix+'.ncnn.param') or net.load_model(prefix+'.ncnn.bin'):
                raise RuntimeError(f'Cannot load ncnn graph {prefix}')
            self.nets[name] = net

    def run(self, name, *arrays):
        # Keep the float32 copies alive until cloned: a Mat only borrows its array's memory.
        arrays = [np.ascontiguousarray(array, np.float32) for array in arrays]
        mats = [self.ncnn.Mat(array).clone() for array in arrays]
        extractor = self.nets[name].create_extractor()
        for i, mat in enumerate(mats):
            extractor.input(f'in{i}', mat)
        status, output = extractor.extract('out0')
        if status:
            raise RuntimeError(f'ncnn {name} failed with {status}')
        return np.array(output).copy()


@dataclass
class _Event:
    camera: int
    capture: float
    arrival: float
    raw: np.ndarray      # [2,21,11], undetected joints zeroed
    valid: np.ndarray    # [2,21]
    tokens: np.ndarray   # [10,dim] finger tokens


class LiteV3Runtime:
    def __init__(self, directory, threads=1, fp16=True, graphs=None):
        """graphs: any object with run(name, *arrays), instead of loading ncnn."""
        directory = Path(directory)
        meta = json.loads((directory/META_FILE).read_text(encoding='utf-8'))
        if meta.get('export_format') != EXPORT_FORMAT:
            raise ValueError(f'{directory} is not a HandLiteV3 export of format {EXPORT_FORMAT}')
        self.config = c = saved_config(meta['config'])
        self.graphs = graphs if graphs is not None else NcnnGraphs(directory, GRAPHS, threads, fp16)
        self.keep_s = c.context_s+STREAM_KEEP_MARGIN_S
        self.reset()

    def reset(self):
        self.events = []
        self.newest_capture = [None]*NUM_CAMERAS
        self.last_arrival = None
        self.error_mm = None
        self.anchor_kind = None
        self.in_view_probability = None   # [2] each hand's probability of being inside some camera's view
        # State: each joint's last triangulation and the capture time of the newest detection it used.
        self.prior_point = np.zeros((NUM_HANDS, NUM_JOINTS, 3))
        self.prior_capture = np.zeros((NUM_HANDS, NUM_JOINTS))
        self.prior_known = np.zeros((NUM_HANDS, NUM_JOINTS), bool)

    def __len__(self):
        return len(self.events)

    def push(self, camera, features, valid, capture_time, arrival_time):
        """Encode one arrived camera frame; False (ignored) for a capture no newer than that
        camera's latest. Frames must arrive in order; frames without detections count."""
        camera = check_event(camera, np.shape(features), np.shape(valid))
        capture, arrival = float(capture_time), float(arrival_time)
        if self.last_arrival is not None and arrival < self.last_arrival:
            raise ValueError('Events must be pushed in arrival order')
        self.last_arrival = arrival
        if self.newest_capture[camera] is not None and capture <= self.newest_capture[camera]:
            return False
        self.newest_capture[camera] = capture
        valid = np.array(valid, dtype=bool, copy=True)
        raw = np.where(valid[..., None], np.asarray(features, np.float64)[..., :MODEL_CHANNELS], 0.)
        tokens = np.asarray(self.graphs.run('encoder', direct_features(raw, valid).reshape(NUM_HANDS, -1),
                                            np.eye(NUM_CAMERAS)[camera]))
        self.events.append(_Event(camera, capture, arrival, raw, valid, tokens))
        while self.events and self.events[0].capture < arrival-self.keep_s:
            self.events.pop(0)
        return True

    def query(self, time):
        """Pose [2,21,3] at time (not before the latest arrival)."""
        time = float(time)
        if self.last_arrival is None:
            raise ValueError('No events yet')
        if time < self.last_arrival:
            raise ValueError('Query before the latest arrival')
        c = self.config
        stored = (self.prior_point, self.prior_known, time-self.prior_capture)
        anchor, self.anchor_kind, features, observed = anchor_features(self.events, time, c, stored)
        triangulated = self.anchor_kind == 1
        self.prior_point = np.where(triangulated[..., None], anchor, self.prior_point)
        self.prior_capture = np.where(triangulated, time-observed, self.prior_capture)
        self.prior_known = self.prior_known | triangulated
        slots = select_slot_events(self.events, time, c.slots_per_camera, c.event_span_s)
        present = np.array([event is not None for event in slots])
        tokens = np.stack([event.tokens if event is not None else np.zeros((EVENT_TOKENS, c.dim)) for event in slots])
        ages = np.array([max(time-event.capture, 0.)/TIME_UNIT_S if event is not None else 0. for event in slots])
        output = np.asarray(self.graphs.run('corrector', features, tokens.reshape(-1, c.dim),
                                            np.repeat(present, EVENT_TOKENS).astype(np.float64),
                                            np.repeat(ages, EVENT_TOKENS))).reshape(HAND_JOINTS, -1)
        column = 3
        if c.error_estimate:                                 # next column: expected error, log(1 + mm)
            self.error_mm = np.expm1(output[:, column]).reshape(NUM_HANDS, NUM_JOINTS)
            column += 1
        if c.presence:                                       # next column: the hand's in-view logit (on its joints)
            logits = output[:, column].reshape(NUM_HANDS, NUM_JOINTS)[:, 0]
            self.in_view_probability = 1/(1+np.exp(-logits))
        return anchor+output[:, :3].reshape(NUM_HANDS, NUM_JOINTS, 3)


In [ ]:
%%writefile /content/gigahands_training/hand_tracking/stream.py
"""Arrival-ordered event buffering for streaming inference, separate from model math.

EventStream owns pushed frames, drops stale captures and prunes old events; each model's
stream subclass says how its per-event encoding moves between relative and absolute time
(to_absolute) and how a query is decoded from the stored events (decode)."""
from dataclasses import dataclass
from typing import Any, NamedTuple
import torch
from .contracts import RawEvents
from .constants import STREAM_KEEP_MARGIN_S, check_event
from .events import relative_events


class PendingEvent(NamedTuple):
    features: torch.Tensor
    valid: torch.Tensor
    camera: int
    capture: float
    arrival: float


@dataclass
class StreamCache:
    """Stored events (absolute float64 capture and arrival times) and their encodings:
    any object with map(fn) and append(other, dim)."""
    raw: RawEvents
    encoded: Any

    def select(self, keep):
        return StreamCache({key: value[keep] for key, value in self.raw.items()},
                           self.encoded.map(lambda value: value[keep]))


class EventStream:
    """Encode each accepted camera event once; query any time from the latest arrival on.

    Events must arrive in order. Empty detections replace older camera rays; duplicate or
    older captures are dropped, matching the batch event policy (events.accepted_captures).
    push() owns its input tensors. query()/flush() encode all pending events in one batch.
    """

    def __init__(self, model):
        self.model = model.eval()
        self.config = model.config
        self.device = next(model.parameters()).device
        # Pruning keeps a margin beyond the exact context a new event can reach.
        self.keep_s = self.config.context_s + STREAM_KEEP_MARGIN_S
        self.reset()

    def reset(self):
        self.store: StreamCache | None = None
        self.pending: list[PendingEvent] = []
        self.newest_capture: dict[int, float] = {}

    def __len__(self):
        stored = 0 if self.store is None else len(self.store.raw['camera'])
        return stored + len(self.pending)

    def _latest_arrival(self):
        if self.pending:
            return self.pending[-1].arrival
        return None if self.store is None else float(self.store.raw['arrival'][-1])

    def push(self, camera, features, valid, capture_time, arrival_time):
        """Own a frame; return False when its capture does not advance that camera."""
        arrival_time = float(arrival_time)
        capture_time = float(capture_time)
        camera = check_event(camera, torch.as_tensor(features).shape, torch.as_tensor(valid).shape)
        latest = self._latest_arrival()
        if latest is not None and arrival_time < latest:
            raise ValueError('Events must be pushed in arrival order')
        if capture_time <= self.newest_capture.get(camera, float('-inf')):
            return False
        event = PendingEvent(torch.as_tensor(features, dtype=torch.float32, device=self.device).clone(),
                             torch.as_tensor(valid, dtype=torch.bool, device=self.device).clone(),
                             camera, capture_time, arrival_time)
        self.newest_capture[camera] = capture_time
        self.pending.append(event)
        return True

    def _pending_raw(self):
        return RawEvents(
            features=torch.stack([event.features for event in self.pending]),
            valid=torch.stack([event.valid for event in self.pending]),
            camera=torch.tensor([event.camera for event in self.pending], device=self.device),
            capture=torch.tensor([event.capture for event in self.pending], dtype=torch.float64, device=self.device),
            arrival=torch.tensor([event.arrival for event in self.pending], dtype=torch.float64, device=self.device))

    @torch.inference_mode()
    def flush(self):
        """Encode pending events transactionally: a failure leaves the buffer intact."""
        if not self.pending:
            return
        origin = self.pending[-1].arrival
        raw = self._pending_raw()
        old = self.store
        if old is not None:
            keep = old.raw['capture'] >= origin-self.keep_s
            if not keep.all():
                old = old.select(keep)
            raw = {key: torch.cat((old.raw[key], value)) for key, value in raw.items()}
        count = len(raw['camera'])
        targets = torch.arange(count-len(self.pending), count, device=self.device)
        encoded = self.model.encode_events(relative_events(raw, origin), targets).map(lambda value: value[0])
        encoded = self.to_absolute(encoded, origin)
        if old is not None:
            encoded = old.encoded.append(encoded, dim=0)
        self.store = StreamCache(raw, encoded)
        self.pending.clear()

    @torch.inference_mode()
    def query(self, time):
        """Pose [2,21,3] at time (not before the latest arrival)."""
        time = float(time)
        latest = self._latest_arrival()
        if latest is None:
            raise ValueError('No events yet')
        if time < latest:
            raise ValueError('Query before the latest arrival')
        self.flush()
        return self.decode(time)

    def to_absolute(self, encoded, origin):
        """A new unbatched encoding with its times on origin -> absolute float64 times."""
        return encoded

    def decode(self, time):
        raise NotImplementedError


In [ ]:
# 5. 데이터 분리·파일 크기와 소스 확인
import importlib, uuid
importlib.invalidate_caches()
SOURCE_HASHES = {'training/__init__.py': '7c8c03e7f7bc8c274605e0bb27d727a603f1df54d45069d05dcf3032015bc8c2', 'training/train.py': '39bc8cb0f49090f61136b2b88a9279c4028bd54df92ade9a61fff461d3b9b06f', 'training/evaluate.py': '71741970fc355994139267028070f36a1b1c9d030bd2e6e950d61c1a17d77d98', 'training/predict.py': 'ea5c1f2c2506e625d5d4d567a05e1e77f8c7f4d43562f766c4be0b58602b2ddb', 'training/export.py': 'f8dec4f1b3515d5c51944f31a850b604312b1565006db3dae7c279b954aac5ff', 'hand_tracking/__init__.py': '166e0ecdd3de289e24bb3e1b27966969d9e37e6ae1e6599069c69d4ce2b7c07e', 'hand_tracking/accelerator.py': '4e5b0377899d965d89f4710911e1cc6b4eb01912002414268712cd3e81d7a73d', 'hand_tracking/checkpoints.py': '3192e90ffd7558b2a7946350c0ac703253b0d0dffc3b87ada7a04ea3722c12c8', 'hand_tracking/config.py': 'f13640ada4b6c104f44d227243c0b6b5b6a038aec0a676bf275cd258c7dec828', 'hand_tracking/constants.py': '172cfe1cfd43bdeb8f35b895bcfc11f726ef42207970233e75153ea7fa2dab36', 'hand_tracking/contracts.py': 'd616c5b0f7e70d29ddbdc332ff56183e4b663e1004fe48223677f3dc9ee1ad6f', 'hand_tracking/data.py': 'a4a8aeca3f566bbd7e8050a401f2ab56ca704551323f99e147b62f91d153d6c1', 'hand_tracking/deploy.py': '4a3898521b90915232d9d0bebd311cef2c7ed01e1cafd36df0f619cee555ed39', 'hand_tracking/direct.py': 'e7ed12c65cc6c56363c3d5475709cd329ba26ba05b12c4dfe3260109afad27f6', 'hand_tracking/direct_export.py': '7661cc07bb853210e866098dc7c3297e7b12bfc8d87bace8308589132b755b2c', 'hand_tracking/direct_runtime.py': 'c5bd463f22a1eae416fbb54611af2aae0b0c9bdc114c2ca70ccbdb988aa67583', 'hand_tracking/engine.py': '1890bcf7681be1e6c116eeddb53198dea6ea1a0eb2bbf38c41790f33c96dbc25', 'hand_tracking/events.py': '14c47633bbc2681bee7c815615553b51105be6e63181d6aefb7c5e02fbc83a7d', 'hand_tracking/geometry.py': '330ec6f27099df4efb2908be4bdff2b271acc0f776376d99a4933dedeba455fc', 'hand_tracking/layers.py': 'b223f991eb047405c468d9a27a905f04fd1cffeab7acdf60e3ce8ca735dee939', 'hand_tracking/model.py': '40d04cbb7266db2115e643b0399e2eb696e549f178afbcc6b05a7d06ac36eff7', 'hand_tracking/networks.py': '56f77b5cd6621b175a8cfc0c77f13b645f38fbd278a6f917212f4509e069c4ed', 'hand_tracking/objectives.py': '103f3bace0de908136b3b45570a63a44013b8da987c94d50834f7f0282721f05', 'hand_tracking/runtime.py': '1f003210bb09f547301ef4350b08eb45d3346a03c929497c705a0df085b62276', 'hand_tracking/stream.py': 'f588c248fe519786661a157e06353a7f29d9b14a7a329490665335b998faf9c0'}
for name, expected in SOURCE_HASHES.items():
    actual = hashlib.sha256((WORK_DIR / name).read_text().encode()).hexdigest()
    assert actual == expected, f'소스 변경 감지: {name}. 변경 실험은 노트북과 실행 이름을 새로 만드세요.'
from hand_tracking.checkpoints import build_model
from hand_tracking.config import ARCHITECTURES
from hand_tracking.data import HandWindows, read_manifest
rows = read_manifest(DATA_DIR)  # 참가자/원본 train-val 중복이면 오류
for row in tqdm(rows, desc='Check dataset'):
    assert (DATA_DIR / row['file']).stat().st_size == row['bytes'], row['file']
for split in ('train', 'val'):
    selected = [r for r in rows if r['split'] == split]
    print(split, 'clips:', len(selected), 'participants:', len({r['participant'] for r in selected}))
heads = dict(error_estimate=ERROR_ESTIMATE, presence=PRESENCE) if ARCH == 'litev3' else {}
cfg = ARCHITECTURES[ARCH](dropout=DROPOUT, **heads, **{key[2:].replace('-', '_'): value for key, value in MODEL_OPTIONS.items()})
# 시야 판정 확인: 클립마다 검출된 관절의 u·v와 명목 ray로 카메라별 픽셀 변환(u = a·x + b)을 맞추고, 정답 관절을 실제
# 카메라로 투영해 시야 안/밖을 정합니다(data.hands_in_view). 맞춤 잔차가 크면 이 가정(핀홀, ray = 픽셀 @ R)이 틀린 것입니다.
from hand_tracking.data import pixel_model, read_clip
residuals, out_share = [], []
# 시야 밖 손은 전체 query의 약 1%이고 일부 클립에 몰려 있어 20개 표본으로는 0%가 나올 수 있으므로 100개를 봅니다.
for index in tqdm(np.random.default_rng(1).choice(len(rows), min(100, len(rows)), replace=False), desc='In-view check'):
    with np.load(DATA_DIR / rows[index]['file'], allow_pickle=False) as clip:
        _, rms = pixel_model(clip['features'], clip['input_mask'], clip['nominal_rotations'])
    residuals += [value for value in rms if np.isfinite(value)]
    in_view = read_clip(DATA_DIR / rows[index]['file'])['hand_in_view']
    out_share.append(1-in_view.mean())
print(f'픽셀 변환 맞춤 잔차(화면 폭 대비): 중앙 {np.median(residuals):.4f}, 최대 {np.max(residuals):.4f}; '
      f'시야 밖 손 비율(query 기준): 평균 {100*np.mean(out_share):.1f}%, 클립별 최대 {100*np.max(out_share):.1f}%')
if not residuals or np.max(residuals) > 0.01:
    print('경고: 픽셀 변환 맞춤 잔차가 화면의 1%를 넘습니다. 시야 판정이 부정확할 수 있으니 학습 전에 확인하세요.')
# 고정값 확인(값은 바꾸지 않음): 카메라 fps가 20 이하라 직선 맞춤 창에 카메라당 최대 fit_span_s*20 프레임,
# 구간당 이벤트는 MAX_EVENTS 안에 들어와야 합니다. 넘치면 가장 오래된 이벤트부터 버려집니다.
probe = HandWindows(DATA_DIR, 'val', windows_per_clip=4, max_clips=30, context_s=cfg.context_s, max_events=4096)
counts, intervals = [], []
for sample in probe:
    present = sample['event_present']
    counts.append(int(present.sum()))
    for camera in range(3):
        capture = np.sort(sample['event_capture'][present & (sample['event_camera'] == camera)].numpy())
        if len(capture) > 1:
            intervals.append(np.diff(capture))
counts = np.array(counts)
fps = 1/float(np.median(np.concatenate(intervals)))
fit = f'직선 맞춤 창 {cfg.fit_span_s:g}초에 카메라당 약 {fps*cfg.fit_span_s:.0f}프레임; ' if ARCH == 'litev3' else ''
print(f'데이터의 카메라 fps 약 {fps:.1f} ({fit}'
      f'손가락 토큰 창 {cfg.event_span_s:g}초에 약 {fps*cfg.event_span_s:.0f}프레임, slots_per_camera={cfg.slots_per_camera})')
print(f'구간당 이벤트 수: 중앙 {int(np.median(counts))}, 99% {int(np.percentile(counts, 99))}, 최대 {counts.max()} '
      f'(MAX_EVENTS={MAX_EVENTS}, context {cfg.context_s:.2f}s)')
if np.percentile(counts, 99) > MAX_EVENTS:
    print(f'경고: 구간 이벤트 수가 MAX_EVENTS를 넘습니다. 설정 셀에서 {int(np.percentile(counts, 99))} 이상으로 늘리세요.')
temporary_model = build_model(cfg)
print('parameters:', f'{sum(p.numel() for p in temporary_model.parameters()):,}')
del temporary_model


def _stop(process):
    if process.poll() is None:
        process.terminate()
        try:
            process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()


def run_python(module, *arguments):
    command = [sys.executable, '-u', '-m', module, *map(str, arguments)]
    print(' '.join(command), flush=True)
    process = subprocess.Popen(command, cwd=WORK_DIR, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in process.stdout:
            print(line, end='', flush=True)
        code = process.wait()
        if code:
            raise subprocess.CalledProcessError(code, command)
    except BaseException:
        _stop(process)
        raise


def run_training(arguments, epochs, batch_size, done=0):
    """run_python('training.train') with progress bars read from its log: all epochs, and
    this epoch's train/val batches (moved every 50 batches, as train.py logs; the batch totals
    are upper bounds, so a bar fills at the phase's end). Other lines print as usual."""
    import math, re
    command = [sys.executable, '-u', '-m', 'training.train', *map(str, arguments)]
    print(' '.join(command), flush=True)
    overall = tqdm(total=epochs, initial=done, desc='전체', unit='epoch')
    phases = {name: tqdm(total=1, desc=f'{name} (epoch {done+1})', unit='batch') for name in ('train', 'val')}
    sizes = {}
    def finish(bar):
        if bar.n < bar.total:
            bar.update(bar.total-bar.n)
    process = subprocess.Popen(command, cwd=WORK_DIR, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in process.stdout:
            if sizes_line := re.search(r'train_windows/epoch<=([\d,]+) val_windows<=([\d,]+)', line):
                for name, windows in zip(phases, sizes_line.groups()):
                    sizes[name] = max(math.ceil(int(windows.replace(',', ''))/batch_size), 1)
                    phases[name].reset(total=sizes[name])
                print(line, end='', flush=True)
            elif progress := re.match(r'(train|val) batch=(\d+) samples=\d+ MPJPE=([\d.]+)mm(.*)', line):
                name, batches = progress[1], int(progress[2])
                if name == 'val':
                    finish(phases['train'])
                bar = phases[name]
                bar.update(min(batches, bar.total)-bar.n)
                bar.set_postfix_str(f'MPJPE {progress[3]}mm {progress[4].strip()}'.strip())
            elif ended := re.match(r'epoch=(\d+)/\d+ (.*)', line):
                epoch = int(ended[1])
                overall.update(epoch-overall.n)
                overall.set_postfix_str(ended[2].strip())
                print(line, end='', flush=True)
                for name, bar in phases.items():   # ready for the next epoch
                    if epoch < epochs:
                        bar.reset(total=sizes.get(name, 1))
                        bar.set_description(f'{name} (epoch {epoch+1})')
                    else:
                        finish(bar)
            else:
                print(line, end='', flush=True)
        code = process.wait()
        if code:
            raise subprocess.CalledProcessError(code, command)
    except BaseException:
        _stop(process)
        raise
    finally:
        for bar in (overall, *phases.values()):
            bar.close()


## TPU 동작 확인
본학습 전에 실제 데이터의 작은 배치로 PyTorch/XLA·컴파일·역전파·체크포인트 저장(CPU로 옮겨 저장)을 확인합니다. 성능 평가가 아니며 본학습 가중치로 쓰지 않습니다. 첫 배치는 XLA 컴파일로 시간이 걸립니다.


In [ ]:
# 6. TPU smoke test — 본학습 실행 폴더와 분리
smoke_dir = WORK_DIR / ('smoke_' + uuid.uuid4().hex[:8])
run_python('training.train',
           '--data', DATA_DIR, '--output', smoke_dir, '--epochs', 1,
           '--batch-size', min(BATCH_SIZE, 2), *MODEL_ARGS, *LOSS_ARGS,
           '--max-clips', 4, '--max-train-batches', 2, '--max-val-batches', 2, '--train-queries', TRAIN_QUERIES,
           '--windows-per-clip', 2, '--val-windows-per-clip', 2,
           '--workers', 0, '--device', DEVICE)
# TPU에서 저장한 체크포인트가 torch_xla 없이(CPU) 열리는지 확인 — 예측·내보내기 셀이 CPU에서 읽습니다.
from hand_tracking.checkpoints import load_checkpoint
smoke = load_checkpoint(smoke_dir / 'last.pt')
assert all(p.device.type == 'cpu' for p in smoke.model.parameters())
del smoke
print('TPU smoke test passed.')


## 본학습 / 재개
- 학습 클립마다 epoch당 최대 16개 무작위 윈도입니다. 윈도는 query 시각 16개와, 첫 query보다 1.45초(직전 위치 탐색 1.0초 + 직선 맞춤 0.2초 + 도착 여유 0.25초) 전부터 마지막 query까지 도착한 이벤트(최대 `MAX_EVENTS`개)입니다. 학습은 윈도의 마지막 query 4개만 씁니다.
- 검증 구간은 고정입니다. MPJPE는 마지막 query의 3D 관절 오차(mm)입니다. 정답 좌표계는 `rig`(카메라 배치를 실제로 놓인 대로 본 좌표계, 관측할 수 없는 rig 전체 어긋남 제거)이고, 참 월드 좌표 오차는 `mpjpe_world_mm`으로 함께 기록합니다.
- 로그: `rel=`은 손목 기준 상대 좌표 오차(손 모양), `error_miss=`는 예상 오차가 실제 오차에서 평균 몇 mm 빗나가는지입니다. 다음 줄 `kinds`는 기준점 종류별 비율 / MPJPE / (보정 전 기준점만의 오차)입니다.
- **매 epoch 완료 시** Drive에 `last.pt`, 개선 시 `best.pt`를 저장합니다. 같은 설정·같은 `RUN_NAME`으로 다시 실행하면 마지막 완료 epoch부터 재개합니다. 설정을 바꾸면 재개가 거부되니 새 `RUN_NAME`으로 시작하세요.
- 첫 epoch가 끝나기 전에 중단되어 `last.pt`가 없으면 새 `RUN_NAME`을 쓰세요. 기존 결과는 자동으로 지우지 않습니다.
- 학습 증강은 없습니다. 카메라 배치가 고정이라 그 배치의 월드 좌표를 그대로 배웁니다.


In [ ]:
# 7. Train / Resume — TPU(XLA)에서 학습, Drive에 epoch마다 저장
# 첫 train/val 배치는 XLA 컴파일로 수십 초~수 분 걸립니다(로그의 'first XLA step'). 이후 같은 shape는 재사용합니다.
EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)
source_file = EXPERIMENT_DIR / 'source_hashes.json'
if source_file.is_file():
    assert json.loads(source_file.read_text()) == SOURCE_HASHES, '이 실행의 모델 소스가 달라졌습니다. 새 RUN_NAME을 사용하세요.'
else:
    source_file.write_text(json.dumps(SOURCE_HASHES, indent=2))
    snapshot = EXPERIMENT_DIR / 'source_snapshot'
    snapshot.mkdir(exist_ok=True)
    for name in SOURCE_HASHES:
        (snapshot / name).parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(WORK_DIR / name, snapshot / name)

if not CACHE_DIR.exists():
    assert shutil.disk_usage('/content').free > 14003487420, '전처리 캐시(데이터의 약 2배) 공간이 부족합니다.'
resume = (RUN_DIR / 'last.pt').is_file()
if RUN_DIR.exists() and not resume:
    raise RuntimeError('완료된 체크포인트가 없는 기존 실행 폴더입니다. RUN_NAME을 새 이름으로 바꾸고 설정 셀부터 다시 실행하세요.')
arguments = [
    '--data', DATA_DIR, '--output', RUN_DIR, '--epochs', EPOCHS, '--batch-size', BATCH_SIZE, '--lr', LEARNING_RATE,
    *MODEL_ARGS, *LOSS_ARGS,
    '--windows-per-clip', WINDOWS_PER_CLIP, '--val-windows-per-clip', VAL_WINDOWS_PER_CLIP, '--train-queries', TRAIN_QUERIES,
    '--workers', WORKERS, '--prefetch', PREFETCH, '--threads', THREADS, '--seed', SEED, '--device', DEVICE,
    '--cache-dir', CACHE_DIR,
]
done = 0
if resume:
    arguments.append('--resume')
    print('Resume:', RUN_DIR / 'last.pt')
    done = len({json.loads(line)['epoch'] for line in (RUN_DIR / 'metrics.jsonl').read_text().splitlines() if line.strip()})
run_training(arguments, EPOCHS, BATCH_SIZE, done)
print('체크포인트:', RUN_DIR / 'best.pt')


In [ ]:
# 8. 학습 곡선
import matplotlib.pyplot as plt
records = [json.loads(line) for line in (RUN_DIR / 'metrics.jsonl').read_text().splitlines() if line.strip()]
# 중단 시 마지막 완료 epoch의 기록이 반복될 수 있어 epoch별 마지막 행 표시
records = list({r['epoch']: r for r in records}.values())
records.sort(key=lambda r: r['epoch'])
epochs = [r['epoch'] for r in records]
metric = lambda split, key, scale=1: [np.nan if r[split].get(key) is None else r[split][key] * scale for r in records]
kind_metric = lambda key, name: [np.nan if r['val'][key].get(name) is None else r['val'][key][name] for r in records]
fig, axes = plt.subplots(1, 4, figsize=(21, 4))
for split in ('train', 'val'):
    axes[0].plot(epochs, metric(split, 'mpjpe_mm'), label=split)
    axes[1].plot(epochs, metric(split, 'pck20', 100), label=split)
    axes[2].plot(epochs, metric(split, 'mpjpe_rel_mm'), label=split)
for name in ('triangulated', 'ray', 'prior'):             # val: per anchor kind, corrected vs the anchor alone
    line, = axes[3].plot(epochs, kind_metric('kind_mpjpe_mm', name), label=name)
    axes[3].plot(epochs, kind_metric('kind_anchor_mpjpe_mm', name), linestyle=':', color=line.get_color())
axes[0].set(title='MPJPE', xlabel='Epoch', ylabel='mm')
axes[1].set(title='PCK @ 20 mm', xlabel='Epoch', ylabel='%')
axes[2].set(title='Wrist-relative MPJPE (hand shape)', xlabel='Epoch', ylabel='mm')
axes[3].set(title='Val MPJPE by anchor kind (dotted: anchor alone)', xlabel='Epoch', ylabel='mm')
for ax in axes:
    ax.grid(alpha=.3)
    ax.legend()
fig.tight_layout()
fig.savefig(EXPERIMENT_DIR / 'learning_curves.png', dpi=150)
plt.show()


## 검증
고정 검증 윈도에서 `best.pt`를 평가합니다. 기본은 모든 검증 클립의 최대 4개 윈도입니다. `EVAL_WINDOWS_PER_CLIP=0`이면 전체 검증 윈도를 평가합니다. 검증은 모델 선택에 쓰였으므로 독립된 최종 test 성능은 아닙니다.


In [ ]:
# 9. 검증 평가 (고정 검증 구간, TPU)
EVAL_WINDOWS_PER_CLIP = 4
assert (RUN_DIR / 'best.pt').is_file(), '먼저 학습 체크포인트를 생성하세요.'
eval_dir = EXPERIMENT_DIR / ('eval_' + uuid.uuid4().hex[:8])
eval_dir.mkdir()
run_python('training.evaluate', '--checkpoint', RUN_DIR / 'best.pt', '--output', eval_dir / 'model.json',
           '--data', DATA_DIR, '--windows-per-clip', EVAL_WINDOWS_PER_CLIP, '--batch-size', BATCH_SIZE,
           '--workers', WORKERS, '--device', DEVICE)
report = json.loads((eval_dir / 'model.json').read_text())
print(f"MPJPE: {report['mpjpe_mm']:.2f} mm (rig 좌표계), 참 월드 좌표: {report['mpjpe_world_mm']:.2f} mm")
print(f"PCK@20mm: {report['pck20']*100:.1f}%")
print(f"손목 기준 MPJPE(손 모양): {report['mpjpe_rel_mm']:.2f} mm")
if report.get('presence_accuracy') is not None:
    print(f"시야 안/밖 판정 정확도: {report['presence_accuracy']*100:.1f}% (시야 밖 손 비율 {report['out_of_view_rate']*100:.1f}%, 마지막 query)")
if report.get('error_miss_mm') is not None:
    print(f"예상 오차의 평균 빗나감: {report['error_miss_mm']:.2f} mm (작을수록 신뢰도가 정확)")
names = {'triangulated': '삼각측량 기준점', 'ray': '한 ray + 직전 삼각측량 깊이', 'prior': '직전 삼각측량 위치(ray 없음)',
         'none': '기준점 없음(직전 삼각측량도 없음)'}
print('기준점 종류별 (마지막 query, 정답 있는 관절):')
for kind, name in names.items():
    rate, mm = report['kind_rate'].get(kind), report['kind_mpjpe_mm'].get(kind)
    alone = report['kind_anchor_mpjpe_mm'].get(kind)
    if rate is not None:
        print(f"  {name}: {rate*100:.1f}%" + ('' if mm is None else f", MPJPE {mm:.2f} mm")
              + ('' if alone is None else f" (보정 전 기준점만 {alone:.2f} mm)"))
print('결과 저장:', eval_dir)


In [ ]:
# 10. 검증 클립 1개 예측 및 3D 비교
import matplotlib.pyplot as plt
from hand_tracking.objectives import BONES
chosen = next(row for row in rows if row['split'] == 'val' and row['windows'] > 0)
prediction_file = EXPERIMENT_DIR / ('prediction_' + uuid.uuid4().hex[:8] + '.npz')
run_python('training.predict', '--checkpoint', RUN_DIR / 'best.pt',
           '--input', DATA_DIR / chosen['file'], '--output', prediction_file)
with np.load(prediction_file, allow_pickle=False) as data:
    predicted = data['predicted_xyz_cm'].copy()
    target = data['target_xyz'] * float(data['world_unit_cm'])
    valid = data['target_mask'].copy()
fig = plt.figure(figsize=(8, 7))
ax = fig.add_subplot(111, projection='3d')
for hand, color in enumerate(('tab:blue', 'tab:orange')):
    for a, b in BONES:
        if valid[hand, a] and valid[hand, b]:
            ax.plot(*target[hand, [a,b]].T, color=color, linestyle='--', alpha=.6)
        ax.plot(*predicted[hand, [a,b]].T, color=color)
ax.set(xlabel='X (cm)', ylabel='Y (cm)', zlabel='Z (cm)', title='Prediction: solid / Ground truth: dashed')
combined = np.concatenate((predicted.reshape(-1,3), target[valid]), axis=0)
center = (combined.min(0)+combined.max(0))/2
radius = max(float(np.ptp(combined,axis=0).max())/2, 1.)
ax.set_xlim(center[0]-radius,center[0]+radius)
ax.set_ylim(center[1]-radius,center[1]+radius)
ax.set_zlim(center[2]-radius,center[2]+radius)
ax.set_box_aspect((1,1,1))
plt.show()
print('원본:', chosen['source'])
print('저장:', prediction_file)


## 배포용 내보내기 (선택)
`training/export.py`가 신경망을 **ncnn**으로 내보내고(pnnx 변환, 그래프 `encoder`·`corrector`), 라즈베리 파이용 numpy 런타임(`hand_tracking/runtime.py`)의 결과를 검증 클립에서 PyTorch 스트림과 비교합니다. 결과 폴더 전체를 장치로 복사하면 PyTorch 없이 `numpy`와 `ncnn`만으로 추론합니다. 10번 셀(예측)의 검증 클립을 쓰므로 그 셀을 먼저 실행하세요.


In [ ]:
# 11. 배포용 내보내기 (ncnn) + 런타임 확인
missing = [name for name in ('ncnn', 'pnnx') if importlib.util.find_spec(name) is None]
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', *missing], check=True)
export_target = EXPERIMENT_DIR / ('deploy_' + uuid.uuid4().hex[:8])
run_python('training.export', '--checkpoint', RUN_DIR / 'best.pt', '--output', export_target,
           '--check-input', DATA_DIR / chosen['file'])
print('내보내기:', export_target)


## 저장 위치 및 다시 시작하기
- `MyDrive/GigaHands/runs/<RUN_NAME>/checkpoints/best.pt`: 검증 오차가 가장 낮은 가중치 (CPU 텐서로 저장되어 GPU·CPU에서도 그대로 열림)
- `last.pt`: 마지막 완료 epoch의 모델·optimizer·scheduler·난수 상태(TPU 난수 포함)
- `run.json`, `metrics.jsonl`: 설정과 학습 기록
- 상위 실행 폴더: 소스 사본, 학습 곡선, 검증 결과(`eval_*/`), 예측 결과, 배포용 ncnn 모델(`deploy_*/`)

Colab 재연결 시 위에서부터 실행하고 같은 `RUN_NAME`과 설정을 쓰세요. TPU smoke test는 생략할 수 있습니다. 7번 셀이 `last.pt`를 찾아 자동 재개합니다(끝나지 않은 epoch는 처음부터 다시 학습). 재개할 때도 첫 배치는 다시 컴파일합니다.

**실행 검증 범위:** 내장 코드는 로컬 CPU 테스트로 검증했습니다. 이 노트북을 만들 때 실제 TPU에서 실행한 것은 아니니 6번 TPU smoke test로 먼저 확인하세요. 매 배치가 첫 배치만큼 느리면 배치마다 shape가 바뀌어 다시 컴파일한다는 뜻입니다.
